# Preparación e integración de datos históricos de la LPF
En este cuaderno de trabajo (*notebook*) muestro el proceso de carga, limpieza, validación y preparación inicial de los datos con los que construiré el sistema predictivo para la Liga Panameña de Fútbol.
La información principal proviene de los registros históricos de partidos y equipos descargados de FootyStats. Más adelante, integraré de forma independiente los datos correspondientes a la temporada 2017/2018 utilizando una fuente histórica complementaria para completar la base de datos.
Todo el trabajo realizado en esta etapa servirá como punto de partida para el Análisis Exploratorio de Datos, la creación de las variables del modelo y el entrenamiento de los algoritmos de predicción.

## Celda 2 Importaciones
Para mantener una estructura limpia y ordenada en la ejecución de este cuaderno, agrupe únicamente las librerías necesarias para las tareas de carga, limpieza y manipulación tabular.

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.3
NumPy: 2.4.6


## Celda 3 Definición de rutas
Establezco la estructura de directorios utilizando la librería `pathlib` para asegurar la compatibilidad de las rutas independientemente del sistema operativo. 

In [2]:
from pathlib import Path

ROOT = Path("..").resolve()

PARTIDOS_DIR = ROOT / "data" / "raw" / "partidos"
EQUIPOS_DIR = ROOT / "data" / "raw" / "equipos"
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Proyecto:", ROOT)
print("Partidos:", PARTIDOS_DIR)
print("Equipos:", EQUIPOS_DIR)
print("Procesados:", PROCESSED_DIR)

print("\n¿Existe carpeta partidos?", PARTIDOS_DIR.exists())
print("¿Existe carpeta equipos?", EQUIPOS_DIR.exists())

Proyecto: C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF
Partidos: C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\raw\partidos
Equipos: C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\raw\equipos
Procesados: C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\processed

¿Existe carpeta partidos? True
¿Existe carpeta equipos? True


## Celda 4 Verificación de archivos disponibles
Reviso y listo los archivos `.csv` almacenados en las carpetas de origen para confirmar la presencia del conjunto completo de datos antes de proceder con su carga.

In [3]:
archivos_partidos = sorted(PARTIDOS_DIR.glob("*.csv"))
archivos_equipos = sorted(EQUIPOS_DIR.glob("*.csv"))

print(f"Archivos de partidos encontrados: {len(archivos_partidos)}")
print(f"Archivos de equipos encontrados: {len(archivos_equipos)}")

print("\nArchivos de partidos:")
for archivo in archivos_partidos:
    print(" -", archivo.name)

print("\nArchivos de equipos:")
for archivo in archivos_equipos:
    print(" -", archivo.name)

Archivos de partidos encontrados: 12
Archivos de equipos encontrados: 12

Archivos de partidos:
 - panama-lpf-matches-2014-to-2015-stats.csv
 - panama-lpf-matches-2015-to-2016-stats.csv
 - panama-lpf-matches-2016-to-2017-stats.csv
 - panama-lpf-matches-2018-to-2019-stats.csv
 - panama-lpf-matches-2020-to-2020-stats.csv
 - panama-lpf-matches-2021-to-2021-stats.csv
 - panama-lpf-matches-2022-to-2022-stats.csv
 - panama-lpf-matches-2023-to-2023-stats.csv
 - panama-lpf-matches-2024-to-2024-stats.csv
 - panama-lpf-matches-2025-to-2025-stats.csv
 - panama-lpf-matches-2026-to-2026-stats.csv
 - panama-lpf-matches-2026-to-2027-stats.csv

Archivos de equipos:
 - panama-lpf-teams-2014-to-2015-stats.csv
 - panama-lpf-teams-2015-to-2016-stats.csv
 - panama-lpf-teams-2016-to-2017-stats.csv
 - panama-lpf-teams-2018-to-2019-stats.csv
 - panama-lpf-teams-2020-to-2020-stats.csv
 - panama-lpf-teams-2021-to-2021-stats.csv
 - panama-lpf-teams-2022-to-2022-stats.csv
 - panama-lpf-teams-2023-to-2023-stats.cs

## Celda 5 Función para la extracción de la temporada
Defino una función auxiliar que utiliza expresiones regulares sobre los nombres de los archivos fuente descargados de FootyStats. Esto me permite parsear e identificar automáticamente la temporada correspondiente a cada archivo `.csv`.

In [4]:
def obtener_temporada(nombre_archivo):
    patron = r"(\d{4})-to-(\d{4})"
    coincidencia = re.search(patron, nombre_archivo)

    if coincidencia:
        inicio = coincidencia.group(1)
        fin = coincidencia.group(2)

        if inicio == fin:
            return inicio

        return f"{inicio}/{fin}"

    return "Desconocida"

In [5]:
for archivo in archivos_partidos:
    print(
        archivo.name,
        "→",
        obtener_temporada(archivo.name)
    )

panama-lpf-matches-2014-to-2015-stats.csv → 2014/2015
panama-lpf-matches-2015-to-2016-stats.csv → 2015/2016
panama-lpf-matches-2016-to-2017-stats.csv → 2016/2017
panama-lpf-matches-2018-to-2019-stats.csv → 2018/2019
panama-lpf-matches-2020-to-2020-stats.csv → 2020
panama-lpf-matches-2021-to-2021-stats.csv → 2021
panama-lpf-matches-2022-to-2022-stats.csv → 2022
panama-lpf-matches-2023-to-2023-stats.csv → 2023
panama-lpf-matches-2024-to-2024-stats.csv → 2024
panama-lpf-matches-2025-to-2025-stats.csv → 2025
panama-lpf-matches-2026-to-2026-stats.csv → 2026
panama-lpf-matches-2026-to-2027-stats.csv → 2026/2027


## Celda 6  Consolidador de partidos
Leo e integro los 12 archivos `.csv` de partidos descargados de FootyStats. Asigno las columnas de metadatos correspondientes (`temporada`, `archivo_origen` y `fuente`) y concateno toda la información en un único DataFrame.

In [6]:
dataframes_partidos = []

for archivo in archivos_partidos:
    # lectura de archivo individual
    df_temp = pd.read_csv(archivo)

    # asignacion de metadatos
    df_temp["temporada"] = obtener_temporada(archivo.name)
    df_temp["archivo_origen"] = archivo.name
    df_temp["fuente"] = "FootyStats"

    dataframes_partidos.append(df_temp)

# consolidacion de dataframes
df_partidos = pd.concat(
    dataframes_partidos,
    ignore_index=True
)

print("Dimensiones del dataset consolidado:")
print(df_partidos.shape)

Dimensiones del dataset consolidado:
(2135, 69)


## Celda 7  Comprobación general

Verifico las dimensiones finales del dataset concatenado y muestro las primeras filas para validar la correcta lectura de las columnas y los metadatos agregados.

In [7]:
print(f"Total de registros: {len(df_partidos):,}")
print(f"Total de columnas: {df_partidos.shape[1]}")

display(df_partidos.head())

Total de registros: 2,135
Total de columnas: 69


,timestamp,date_GMT,status,attendance,home_team_name,away_team_name,referee,Game Week,Pre-Match PPG (Home),Pre-Match PPG (Away),home_ppg,away_ppg,home_team_goal_count,away_team_goal_count,total_goal_count,total_goals_at_half_time,home_team_goal_count_half_time,away_team_goal_count_half_time,home_team_goal_timings,away_team_goal_timings,home_team_corner_count,away_team_corner_count,home_team_yellow_cards,home_team_red_cards,away_team_yellow_cards,away_team_red_cards,home_team_first_half_cards,home_team_second_half_cards,away_team_first_half_cards,away_team_second_half_cards,home_team_shots,away_team_shots,home_team_shots_on_target,away_team_shots_on_target,home_team_shots_off_target,away_team_shots_off_target,home_team_fouls,away_team_fouls,home_team_possession,away_team_possession,Home Team Pre-Match xG,Away Team Pre-Match xG,team_a_xg,team_b_xg,average_goals_per_match_pre_match,btts_percentage_pre_match,over_15_percentage_pre_match,over_25_percentage_pre_match,over_35_percentage_pre_match,over_45_percentage_pre_match,over_15_HT_FHG_percentage_pre_match,over_05_HT_FHG_percentage_pre_match,over_15_2HG_percentage_pre_match,over_05_2HG_percentage_pre_match,average_corners_per_match_pre_match,average_cards_per_match_pre_match,odds_ft_home_team_win,odds_ft_draw,odds_ft_away_team_win,odds_ft_over15,odds_ft_over25,odds_ft_over35,odds_ft_over45,odds_btts_yes,odds_btts_no,stadium_name,temporada,archivo_origen,fuente
0,1405731600,Jul 19 2014 - 1:00am,complete,NaN,Árabe Unido,Alianza,José Antonio Kellys Marquez,1.0,0.0,0.0,1.75,1.00,0,1,1,1,0,1,NaN,15,-1,-1,1,0,6,0,1,0,2,4,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Estadio Rommel Fernández Gutiérrez (Ciudad de ...,2014/2015,panama-lpf-matches-2014-to-2015-stats.csv,FootyStats
1,1405735200,Jul 19 2014 - 2:00am,complete,NaN,San Francisco FC,Tauro,Roberto Moreno Salazar,1.0,0.0,0.0,1.21,1.45,0,0,0,0,0,0,NaN,NaN,-1,-1,1,0,3,0,0,1,1,2,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Estadio Agustín Muquita Sánchez (La Chorrera),2014/2015,panama-lpf-matches-2014-to-2015-stats.csv,FootyStats
2,1405735200,Jul 19 2014 - 2:00am,complete,NaN,Atlético Chiriquí,Plaza Amador,NaN,1.0,0.0,0.0,1.28,1.42,0,0,0,0,0,0,NaN,NaN,-1,-1,3,0,1,0,1,2,0,1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Estadio San Cristóbal (Ciudad de David),2014/2015,panama-lpf-matches-2014-to-2015-stats.csv,FootyStats
3,1405810800,Jul 19 2014 - 11:00pm,complete,NaN,Chepo,Sporting San Miguelito,NaN,1.0,0.0,0.0,1.17,1.45,1,2,3,2,0,2,75,"29,35",-1,-1,2,1,4,0,1,2,2,2,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Estadio Rommel Fernández Gutiérrez (Ciudad de ...,2014/2015,panama-lpf-matches-2014-to-2015-stats.csv,FootyStats
4,1405810800,Jul 19 2014 - 11:00pm,complete,NaN,Chorrillo,CA Independiente de La Chorrera,NaN,1.0,0.0,0.0,1.21,0.80,0,0,0,0,0,0,NaN,NaN,-1,-1,0,0,1,0,0,0,1,0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Nuevo Estadio Maracaná de Panamá (Ciudad de Pa...,2014/2015,panama-lpf-matches-2014-to-2015-stats.csv,FootyStats


In [8]:
print("Registros por temporada:\n")

conteo_temporadas = (
    df_partidos["temporada"]
    .value_counts()
    .sort_index()
)

display(conteo_temporadas)

Registros por temporada:



temporada
2014/2015    190
2015/2016    190
2016/2017    190
2018/2019    194
2020         142
2021         206
2022         206
2023         206
2024         206
2025         206
2026         103
2026/2027     96
Name: count, dtype: int64

## Celda 8  Verificación del estado de los partidos
Identifico las columnas relacionadas con el estado de los encuentros y analizo sus valores únicos para filtrar únicamente los partidos disputados y descartar registros incompletos o suspendidos.

In [9]:
[column for column in df_partidos.columns if "status" in column.lower()]

['status']

In [10]:
conteo_status = df_partidos["status"].value_counts(dropna=False)
display(conteo_status)

status
complete      2037
canceled        50
incomplete      48
Name: count, dtype: int64

In [11]:
porcentaje_status = (
    df_partidos["status"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)
display(porcentaje_status)

status
complete      95.41
canceled       2.34
incomplete     2.25
Name: proportion, dtype: float64

## Celda 9 Separación inicial según estado
Filtro el conjunto de datos conservando únicamente los partidos con estado completado (`complete`), los cuales cuentan con la información requerida para el entrenamiento del modelo. Separamos los registros cancelados e incompletos.

In [12]:
# filtrado por estado del partido
df_complete = df_partidos[
    df_partidos["status"].eq("complete")
].copy()

df_canceled = df_partidos[
    df_partidos["status"].eq("canceled")
].copy()

df_incomplete = df_partidos[
    df_partidos["status"].eq("incomplete")
].copy()

# verificacion de registros por estado
print("Partidos completados:", len(df_complete))
print("Partidos cancelados:", len(df_canceled))
print("Partidos incompletos:", len(df_incomplete))

Partidos completados: 2037
Partidos cancelados: 50
Partidos incompletos: 48


## Celda 10 Identificación y validación temporal
Verifico las columnas relacionadas con la fecha de los encuentros y convierto la información temporal a un formato uniforme. Esta etapa me permite ordenar cronológicamente los partidos y garantizar posteriormente que las características predictivas se construyan únicamente con información anterior al encuentro evaluado.

In [13]:
[column for column in df_complete.columns
 if "date" in column.lower() or "time" in column.lower()]

['timestamp',
 'date_GMT',
 'total_goals_at_half_time',
 'home_team_goal_count_half_time',
 'away_team_goal_count_half_time']

## Celda 10.1  Conversión y estandarización de fechas
Convierto tanto la fecha en formato de texto (`date_GMT`) como el valor numérico en Unix (`timestamp`) a objetos de tipo fecha y hora UTC. Esta doble transformación me permite validar que ambas fuentes temporales coincidan correctamente.

In [14]:
# limpieza basica de texto
df_complete["date_GMT_limpia"] = (
    df_complete["date_GMT"]
    .astype("string")
    .str.strip()
)

# conversion explicita con formato
df_complete["fecha_gmt"] = pd.to_datetime(
    df_complete["date_GMT_limpia"],
    format="%b %d %Y - %I:%M%p",
    errors="coerce",
    utc=True
)

# conversion del timestamp unix
df_complete["fecha_timestamp"] = pd.to_datetime(
    df_complete["timestamp"],
    unit="s",
    errors="coerce",
    utc=True
)

display(
    df_complete[
        [
            "date_GMT",
            "timestamp",
            "fecha_gmt",
            "fecha_timestamp"
        ]
    ].head(10)
)

,date_GMT,timestamp,fecha_gmt,fecha_timestamp
0,Jul 19 2014 - 1:00am,1405731600,2014-07-19 01:00:00+00:00,2014-07-19 01:00:00+00:00
1,Jul 19 2014 - 2:00am,1405735200,2014-07-19 02:00:00+00:00,2014-07-19 02:00:00+00:00
2,Jul 19 2014 - 2:00am,1405735200,2014-07-19 02:00:00+00:00,2014-07-19 02:00:00+00:00
3,Jul 19 2014 - 11:00pm,1405810800,2014-07-19 23:00:00+00:00,2014-07-19 23:00:00+00:00
4,Jul 19 2014 - 11:00pm,1405810800,2014-07-19 23:00:00+00:00,2014-07-19 23:00:00+00:00
5,Jul 26 2014 - 2:00am,1406340000,2014-07-26 02:00:00+00:00,2014-07-26 02:00:00+00:00
6,Jul 26 2014 - 2:00am,1406340000,2014-07-26 02:00:00+00:00,2014-07-26 02:00:00+00:00
7,Jul 26 2014 - 11:00pm,1406415600,2014-07-26 23:00:00+00:00,2014-07-26 23:00:00+00:00
8,Jul 26 2014 - 11:00pm,1406415600,2014-07-26 23:00:00+00:00,2014-07-26 23:00:00+00:00
9,Jul 27 2014 - 9:00pm,1406494800,2014-07-27 21:00:00+00:00,2014-07-27 21:00:00+00:00


## Celda 10.2 Verificación de coincidencia temporal
Comparo ambas variables de fecha recién generadas para confirmar que no existan discrepancias entre el texto GMT y el timestamp Unix en ningún registro del conjunto de datos.

In [15]:
print(
    "date_GMT que no pudieron convertirse:",
    df_complete["fecha_gmt"].isna().sum()
)

print(
    "timestamp que no pudieron convertirse:",
    df_complete["fecha_timestamp"].isna().sum()
)

date_GMT que no pudieron convertirse: 0
timestamp que no pudieron convertirse: 0


In [16]:
comparables = (
    df_complete["fecha_gmt"].notna()
    &
    df_complete["fecha_timestamp"].notna()
)

coinciden_fechas = (
    df_complete.loc[comparables, "fecha_gmt"]
    ==
    df_complete.loc[comparables, "fecha_timestamp"]
)

print("Total de partidos:", len(df_complete))
print("Registros comparables:", comparables.sum())
print("Fechas coincidentes:", coinciden_fechas.sum())
print("Fechas diferentes:", (~coinciden_fechas).sum())

Total de partidos: 2037
Registros comparables: 2037
Fechas coincidentes: 2037
Fechas diferentes: 0


## Celda 10.3 Normalización y ordenamiento cronológico
Establezco la variable temporal canónica `fecha_partido` utilizando el timestamp Unix convertido y ordeno el conjunto de datos de forma cronológica para asegurar la secuencia temporal requerida en los análisis posteriores.

In [17]:
# variable temporal canonica
df_complete["fecha_partido"] = df_complete["fecha_timestamp"]

# ordenamiento cronologico
df_complete = (
    df_complete
    .sort_values("fecha_partido")
    .reset_index(drop=True)
)

print("Fecha inicial:", df_complete["fecha_partido"].min())
print("Fecha final:", df_complete["fecha_partido"].max())
print("Fechas inválidas:", df_complete["fecha_partido"].isna().sum())

display(
    df_complete[
        [
            "temporada",
            "fecha_partido",
            "home_team_name",
            "away_team_name"
        ]
    ].head(10)
)

Fecha inicial: 2014-07-19 01:00:00+00:00
Fecha final: 2026-09-15 01:00:00+00:00
Fechas inválidas: 0


,temporada,fecha_partido,home_team_name,away_team_name
0,2014/2015,2014-07-19 01:00:00+00:00,Árabe Unido,Alianza
1,2014/2015,2014-07-19 02:00:00+00:00,San Francisco FC,Tauro
2,2014/2015,2014-07-19 02:00:00+00:00,Atlético Chiriquí,Plaza Amador
3,2014/2015,2014-07-19 23:00:00+00:00,Chepo,Sporting San Miguelito
4,2014/2015,2014-07-19 23:00:00+00:00,Chorrillo,CA Independiente de La Chorrera
5,2014/2015,2014-07-26 02:00:00+00:00,CA Independiente de La Chorrera,Árabe Unido
6,2014/2015,2014-07-26 02:00:00+00:00,Tauro,Chorrillo
7,2014/2015,2014-07-26 23:00:00+00:00,Alianza,Chepo
8,2014/2015,2014-07-26 23:00:00+00:00,Plaza Amador,San Francisco FC
9,2014/2015,2014-07-27 21:00:00+00:00,Sporting San Miguelito,Atlético Chiriquí


## Celda 11 Verificación de registros duplicados
Analizo la presencia de posibles registros duplicados mediante distintos criterios de identificación. Además de verificar filas completamente repetidas, evalúo la combinación formada por la fecha del encuentro, el equipo local y el equipo visitante como clave compuesta para detectar representaciones múltiples de un mismo partido dentro del conjunto de datos.

In [18]:
# duplicados completos
duplicados_completos = df_complete.duplicated().sum()
print("Filas completamente duplicadas:", duplicados_completos)

# duplicados por clave de partido
duplicados_partido = df_complete.duplicated(
    subset=[
        "fecha_partido",
        "home_team_name",
        "away_team_name"
    ]
).sum()

print("Duplicados por clave (fecha + local + visitante):", duplicados_partido)

Filas completamente duplicadas: 0
Duplicados por clave (fecha + local + visitante): 0


In [19]:
columnas_id = [
    columna
    for columna in df_complete.columns
    if (
        columna.lower() == "id"
        or columna.lower().endswith("_id")
        or columna.lower().startswith("id_")
    )
]

print("Columnas identificadoras encontradas:", columnas_id)

Columnas identificadoras encontradas: []


In [20]:
# Fecha sin componente horario para una segunda validacion
df_complete["fecha_dia"] = df_complete["fecha_partido"].dt.date

duplicados_dia = df_complete.duplicated(
    subset=[
        "fecha_dia",
        "home_team_name",
        "away_team_name"
    ],
    keep=False
)

print(
    "Registros involucrados en posibles duplicados por día + equipos:",
    duplicados_dia.sum()
)

if duplicados_dia.any():
    display(
        df_complete.loc[
            duplicados_dia,
            [
                "temporada",
                "fecha_partido",
                "home_team_name",
                "away_team_name"
            ]
        ].sort_values(
            ["fecha_partido", "home_team_name"]
        )
    )

Registros involucrados en posibles duplicados por día + equipos: 0


## Celda 12 Validación de marcadores
Identifico las columnas de goles disponibles en el dataset para confirmar la presencia de los marcadores tanto al tiempo completo (`home_team_goal_count` y `away_team_goal_count`) como al descanso (`home_team_goal_count_half_time` y `away_team_goal_count_half_time`). Asimismo, busco posibles variables precalculadas sobre el resultado del partido.

In [21]:
[
    columna
    for columna in df_complete.columns
    if "goal_count" in columna.lower()
]

['home_team_goal_count',
 'away_team_goal_count',
 'total_goal_count',
 'home_team_goal_count_half_time',
 'away_team_goal_count_half_time']

In [22]:
[
    columna
    for columna in df_complete.columns
    if "result" in columna.lower()
]

[]

## Celda 12.1 Tipos, nulos y valores negativos
Evalúo la integridad de las columnas clave de goles (`home_team_goal_count`, `away_team_goal_count` y `total_goal_count`). Verifico sus tipos de datos, la presencia de valores nulos y aseguro la ausencia de registros numéricos negativos.

In [23]:
columnas_goles = [
    "home_team_goal_count",
    "away_team_goal_count",
    "total_goal_count"
]

# verificacion de tipos de datos
print("Tipos de datos:")
display(df_complete[columnas_goles].dtypes)

# conteo de valores nulos
print("\nValores nulos:")
display(df_complete[columnas_goles].isna().sum())

# deteccion de valores negativos
print("\nValores negativos:")
for columna in columnas_goles:
    negativos = (df_complete[columna] < 0).sum()
    print(f"{columna}: {negativos}")

Tipos de datos:


home_team_goal_count    int64
away_team_goal_count    int64
total_goal_count        int64
dtype: object


Valores nulos:


home_team_goal_count    0
away_team_goal_count    0
total_goal_count        0
dtype: int64


Valores negativos:
home_team_goal_count: 0
away_team_goal_count: 0
total_goal_count: 0


## Celda 12.2 Comprobación de consistencia en el total de goles
Sumo los goles del equipo local (`home_team_goal_count`) y del equipo visitante (`away_team_goal_count`) para generar un total calculado. Valido este resultado contra la columna `total_goal_count` provista por la fuente para garantizar la coherencia matemática de los marcadores.

In [24]:
# calculo de la suma de goles
df_complete["total_goles_calculado"] = (
    df_complete["home_team_goal_count"]
    +
    df_complete["away_team_goal_count"]
)

# validacion contra la columna oficial
consistencia_goles = (
    df_complete["total_goles_calculado"]
    ==
    df_complete["total_goal_count"]
)

# reporte de resultados
print("Partidos evaluados:", len(df_complete))
print("Totales consistentes:", consistencia_goles.sum())
print("Totales inconsistentes:", (~consistencia_goles).sum())

Partidos evaluados: 2037
Totales consistentes: 2037
Totales inconsistentes: 0


## Celda 12.3  Estadísticas básicas del marcador
Calculo los descriptivos numéricos de las variables de marcadores como un control de integridad preliminar para validar la distribución general y detectar de forma temprana posibles valores atípicos o anomalías numéricas.

In [25]:
# estadisticas descriptivas generales
display(
    df_complete[
        [
            "home_team_goal_count",
            "away_team_goal_count",
            "total_goal_count"
        ]
    ].describe()
)

,home_team_goal_count,away_team_goal_count,total_goal_count
count,2037.000000,2037.000000,2037.000000
mean,1.211586,1.058419,2.270005
std,1.128268,1.047977,1.566078
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000
50%,1.000000,1.000000,2.000000
75%,2.000000,2.000000,3.000000
max,9.000000,6.000000,9.000000


In [26]:
# impresion de marcadores maximos
print(
    "Máximo de goles de un equipo local:",
    df_complete["home_team_goal_count"].max()
)

print(
    "Máximo de goles de un equipo visitante:",
    df_complete["away_team_goal_count"].max()
)

print(
    "Máximo total de goles en un partido:",
    df_complete["total_goal_count"].max()
)

Máximo de goles de un equipo local: 9
Máximo de goles de un equipo visitante: 6
Máximo total de goles en un partido: 9


## Celda 13 Construcción de la variable objetivo
Debido a que los archivos originales no contienen una variable categórica explícita para representar el resultado del encuentro, construyo la variable objetivo a partir de la comparación entre los goles finales del equipo local y visitante. Utilizo las categorías `L` para victoria local, `E` para empate y `V` para victoria visitante.

In [27]:
# condiciones logicas para determinar el resultado
condiciones = [
    df_complete["home_team_goal_count"]
    > df_complete["away_team_goal_count"],

    df_complete["home_team_goal_count"]
    == df_complete["away_team_goal_count"],

    df_complete["home_team_goal_count"]
    < df_complete["away_team_goal_count"]
]

categorias = ["L", "E", "V"]

# asignacion de la variable objetivo
df_complete["resultado"] = np.select(
    condiciones,
    categorias,
    default="ERROR"
)

# verificacion de la distribucion del resultado
print("Distribución inicial de la variable objetivo:")
display(
    df_complete["resultado"]
    .value_counts()
)

Distribución inicial de la variable objetivo:


resultado
L    780
E    634
V    623
Name: count, dtype: int64

## Celda 13.2 Muestra del dataset procesado
Visualizo los primeros 15 registros para confirmar la correcta alineación entre las fechas, los nombres de los equipos, el marcador final y la variable objetivo asignada.

In [28]:
# muestra de verificacion de la variable objetivo
display(
    df_complete[
        [
            "fecha_partido",
            "home_team_name",
            "away_team_name",
            "home_team_goal_count",
            "away_team_goal_count",
            "resultado"
        ]
    ].head(15)
)

,fecha_partido,home_team_name,away_team_name,home_team_goal_count,away_team_goal_count,resultado
0,2014-07-19 01:00:00+00:00,Árabe Unido,Alianza,0,1,V
1,2014-07-19 02:00:00+00:00,San Francisco FC,Tauro,0,0,E
2,2014-07-19 02:00:00+00:00,Atlético Chiriquí,Plaza Amador,0,0,E
3,2014-07-19 23:00:00+00:00,Chepo,Sporting San Miguelito,1,2,V
4,2014-07-19 23:00:00+00:00,Chorrillo,CA Independiente de La Chorrera,0,0,E
5,2014-07-26 02:00:00+00:00,CA Independiente de La Chorrera,Árabe Unido,1,0,L
6,2014-07-26 02:00:00+00:00,Tauro,Chorrillo,0,1,V
7,2014-07-26 23:00:00+00:00,Alianza,Chepo,1,0,L
8,2014-07-26 23:00:00+00:00,Plaza Amador,San Francisco FC,1,2,V
9,2014-07-27 21:00:00+00:00,Sporting San Miguelito,Atlético Chiriquí,2,0,L


## Celda 14 Auditoría de nombres de equipos
Examino las denominaciones de los equipos presentes en los registros históricos de FootyStats para identificar variantes de nombres y establecer posteriormente una nomenclatura canónica. Este procedimiento es necesario antes de integrar los registros complementarios del periodo 2017/2018, debido a que distintas fuentes pueden utilizar denominaciones diferentes para un mismo club.

## Celda 14.1 Obtener todos los equipos únicos
Consolido en un conjunto único todas las denominaciones de equipos registradas en las columnas de local (`home_team_name`) y visitante (`away_team_name`). Aplico una limpieza básica eliminando espacios adicionales y valores nulos para obtener el catálogo completo de clubes presentes en FootyStats.

In [29]:
# extraccion y limpieza de nombres locales
equipos_locales = set(
    df_complete["home_team_name"]
    .dropna()
    .astype(str)
    .str.strip()
)

# extraccion y limpieza de nombres visitantes
equipos_visitantes = set(
    df_complete["away_team_name"]
    .dropna()
    .astype(str)
    .str.strip()
)

# union ordenada de ambos conjuntos
equipos_unicos = sorted(
    equipos_locales.union(equipos_visitantes)
)

print("Total de denominaciones únicas:", len(equipos_unicos))

# impresion del catalogo de equipos
for equipo in equipos_unicos:
    print("-", equipo)

Total de denominaciones únicas: 20
- Alianza
- Atlético Chiriquí
- Atlético Nacional
- Atlético Veragüense
- CA Independiente de La Chorrera
- Chepo
- Chorrillo
- Costa del Este
- Costa del Este Academia
- Herrera
- Plaza Amador
- San Francisco FC
- Santa Gema
- Sporting San Miguelito
- Tauro
- UMECIT
- Universitario
- Unión Coclé
- Veraguas
- Árabe Unido


## Celda 14.2 Frecuencia de aparición de cada denominación
Calculo la presencia total de cada denominación sumando sus participaciones como local y como visitante. Esta métrica permite identificar equipos recurrentes frente a aquellos con denominaciones secundarias o temporales dentro del conjunto de datos.

In [30]:
# conteo de partidos como local y visitante
apariciones_local = (
    df_complete["home_team_name"]
    .value_counts()
)

apariciones_visitante = (
    df_complete["away_team_name"]
    .value_counts()
)

# consolidacion de frecuencia total por equipo
frecuencia_equipos = (
    apariciones_local
    .add(apariciones_visitante, fill_value=0)
    .astype(int)
    .sort_values(ascending=False)
)

display(
    frecuencia_equipos
    .rename("partidos")
    .to_frame()
)

,partidos
home_team_name,
Plaza Amador,383
Tauro,381
San Francisco FC,368
Sporting San Miguelito,368
Árabe Unido,368
Alianza,365
CA Independiente de La Chorrera,304
Universitario,250
Herrera,191


## Celda 14.3 Equipos utilizados por temporada
Muestro el listado de clubes activos agrupados por cada torneo o temporada. Esta segmentación permite detectar ascensos, descensos, cambios de franquicia o discrepancias en las denominaciones a lo largo del tiempo.

In [31]:
# agrupamiento de equipos participando en cada temporada
equipos_por_temporada = {}

for temporada, grupo in df_complete.groupby("temporada"):
    equipos = sorted(
        set(grupo["home_team_name"])
        .union(set(grupo["away_team_name"]))
    )
    equipos_por_temporada[temporada] = equipos

# impresion estructurada por temporada
for temporada, equipos in equipos_por_temporada.items():
    print(f"\n{temporada} ({len(equipos)} equipos)")
    for equipo in equipos:
        print(" -", equipo)


2014/2015 (10 equipos)
 - Alianza
 - Atlético Chiriquí
 - CA Independiente de La Chorrera
 - Chepo
 - Chorrillo
 - Plaza Amador
 - San Francisco FC
 - Sporting San Miguelito
 - Tauro
 - Árabe Unido

2015/2016 (10 equipos)
 - Alianza
 - Atlético Chiriquí
 - Atlético Nacional
 - Chepo
 - Chorrillo
 - Plaza Amador
 - San Francisco FC
 - Sporting San Miguelito
 - Tauro
 - Árabe Unido

2016/2017 (10 equipos)
 - Alianza
 - Atlético Nacional
 - Atlético Veragüense
 - Chorrillo
 - Plaza Amador
 - San Francisco FC
 - Santa Gema
 - Sporting San Miguelito
 - Tauro
 - Árabe Unido

2018/2019 (10 equipos)
 - Alianza
 - CA Independiente de La Chorrera
 - Costa del Este
 - Plaza Amador
 - San Francisco FC
 - Santa Gema
 - Sporting San Miguelito
 - Tauro
 - Universitario
 - Árabe Unido

2020 (10 equipos)
 - Alianza
 - Atlético Chiriquí
 - CA Independiente de La Chorrera
 - Costa del Este
 - Plaza Amador
 - San Francisco FC
 - Sporting San Miguelito
 - Tauro
 - Universitario
 - Árabe Unido

2021 (12 eq

## Celda 14.4 Verificación de la participación total de equipos
Compruebo que la suma de las participaciones de todos los equipos sea equivalente al doble del número de partidos, debido a que cada encuentro debe contener exactamente un equipo local y un equipo visitante.

In [32]:
# verificacion de consistencia matematica de participaciones
total_apariciones = frecuencia_equipos.sum()
apariciones_esperadas = len(df_complete) * 2

print("Apariciones observadas:", total_apariciones)
print("Apariciones esperadas:", apariciones_esperadas)
print(
    "Consistencia:",
    total_apariciones == apariciones_esperadas
)

Apariciones observadas: 4074
Apariciones esperadas: 4074
Consistencia: True


## Celda 15 Obtención complementaria del periodo 2017/2018
Debido a que el periodo 2017/2018 no se encuentra disponible dentro de los archivos históricos obtenidos mediante FootyStats, se utiliza Soccerway como fuente complementaria para recuperar los resultados correspondientes a dicha temporada. La información obtenida será procesada de manera independiente antes de integrarse con el conjunto histórico principal.

In [33]:
import requests
from bs4 import BeautifulSoup

# url objetivo para la temporada 2017/2018
URL_SOCCERWAY = (
    "https://www.soccerway.com/panama/"
    "lpf-2017-2018/results/"
)

# encabezados para la peticion http
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/154.0.0.0 Safari/537.36"
    )
}

# descarga del html
respuesta = requests.get(
    URL_SOCCERWAY,
    headers=headers,
    timeout=30
)

print("Código HTTP:", respuesta.status_code)
print("Tamaño del HTML:", len(respuesta.text))

# parseo con BeautifulSoup
soup = BeautifulSoup(
    respuesta.text,
    "html.parser"
)

print("Título:", soup.title.text.strip() if soup.title else "Sin título")

Código HTTP: 200
Tamaño del HTML: 828576
Título: Soccerway


## Celda 15.1 Respaldo local de la página histórica
Almaceno una copia local del contenido HTML obtenido desde Soccerway con el propósito de conservar una evidencia reproducible de la fuente utilizada durante la extracción. Este respaldo permite repetir posteriormente el procesamiento sin depender de cambios en la estructura de la página web.

In [34]:
import os

# ruta para guardar el archivo html local
ruta_html_local = "soccerway_2017_2018.html"

# guardado del contenido en un archivo de texto
with open(ruta_html_local, "w", encoding="utf-8") as f:
    f.write(respuesta.text)

print(f"Respaldo HTML guardado correctamente en: {ruta_html_local}")
print("Tamaño del archivo guardado:", os.path.getsize(ruta_html_local), "bytes")

Respaldo HTML guardado correctamente en: soccerway_2017_2018.html
Tamaño del archivo guardado: 842026 bytes


### Celda 15.2. Inspección de estructuras tabulares
Examino las tablas disponibles dentro del documento HTML para identificar si los resultados históricos pueden recuperarse directamente mediante estructuras tabulares. Esta inspección permite determinar el método de extracción más adecuado antes de procesar los registros.

In [35]:
!pip install lxml html5lib


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from io import StringIO
import pandas as pd

# deteccion automatica de tablas html mediante pandas
try:
    tablas_soccerway = pd.read_html(
        StringIO(respuesta.text)
    )

    print(
        "Cantidad de tablas HTML detectadas:",
        len(tablas_soccerway)
    )

except ValueError:
    tablas_soccerway = []

    print(
        "No se detectaron tablas HTML interpretables."
    )

No se detectaron tablas HTML interpretables.


In [37]:
for i, tabla in enumerate(tablas_soccerway):
    print(f"\nTABLA {i}")
    print("Dimensiones:", tabla.shape)
    print("Columnas:", list(tabla.columns))

    display(
        tabla.head(5)
    )

### Celda 15.3. Inspección de la estructura HTML de los resultados
Debido a que los resultados no se encuentran contenidos en tablas HTML convencionales, inspecciono la estructura del documento para localizar los elementos donde se almacenan los nombres de los equipos, fechas y marcadores.

In [38]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(respuesta.text, "html.parser")

# Buscar elementos de texto que contengan nombres conocidos de equipos
terminos = [
    "Tauro",
    "Plaza Amador",
    "Santa Gema",
    "Chorrillo",
    "Independiente",
    "Sporting"
]

for termino in terminos:
    encontrados = soup.find_all(
        string=lambda texto: texto and termino.lower() in texto.lower()
    )

    print(f"\n{termino}: {len(encontrados)} coincidencias")

    for elemento in encontrados[:5]:
        print(
            "Etiqueta:",
            elemento.parent.name,
            "| Clase:",
            elemento.parent.get("class"),
            "| Texto:",
            elemento.strip()[:150]
        )


Tauro: 3 coincidencias
Etiqueta: script | Clase: None | Texto: if(!cjs.initialFeeds) {
                cjs.initialFeeds = [];
            }

            cjs.initialFeeds["summary-results"] = {
                data
Etiqueta: script | Clase: None | Texto: if(!cjs.initialFeeds) {
            cjs.initialFeeds = [];
        }

        cjs.initialFeeds['results'] = {
            data: `SA÷1¬~ZA÷PANAMA: LPF 
Etiqueta: script | Clase: None | Texto: var league_archive_data = {"seasons":[{"name":"LPF 2026","url":"\/panama\/lpf\/","winners":{"1":{"id":72055,"name":"Plaza Amador","stageName":null,"ur

Plaza Amador: 3 coincidencias
Etiqueta: script | Clase: None | Texto: if(!cjs.initialFeeds) {
                cjs.initialFeeds = [];
            }

            cjs.initialFeeds["summary-results"] = {
                data
Etiqueta: script | Clase: None | Texto: if(!cjs.initialFeeds) {
            cjs.initialFeeds = [];
        }

        cjs.initialFeeds['results'] = {
            data: `SA÷1¬~ZA÷PAN

### Celda 15.4. Extracción del feed estructurado de resultados
La inspección del documento HTML permitió identificar que Soccerway almacena los resultados históricos dentro de una estructura JavaScript denominada `cjs.initialFeeds['results']`. Debido a que los encuentros no se presentan mediante tablas HTML convencionales, extraigo este bloque de datos para procesarlo de manera estructurada y recuperar posteriormente las fechas, equipos y marcadores de la temporada 2017/2018.

In [39]:
import re

# Localizar el script que contiene el feed de resultados
script_resultados = None

for script in soup.find_all("script"):
    contenido = script.string or script.get_text()

    if contenido and (
        "cjs.initialFeeds['results']" in contenido
        or 'cjs.initialFeeds["results"]' in contenido
    ):
        script_resultados = contenido
        break

print(
    "Script de resultados encontrado:",
    script_resultados is not None
)

Script de resultados encontrado: True


### Celda 15.5. Extracción de la cadena de datos bruta
Aplico una expresión regular sobre el bloque de script para aislar la cadena de datos bruta encapsulada en la propiedad `data`. Esta cadena contiene la secuencia estructurada de registros de la temporada 2017/2018 que posteriormente será parseada para reconstruir los partidos.

In [40]:
# expresión regular para aislar el contenido de la propiedad 'data'
patron_feed = r"data\s*:\s*`([^`]*)`"

coincidencia_feed = re.search(
    patron_feed,
    script_resultados,
    flags=re.DOTALL
)

if coincidencia_feed:
    feed_resultados = coincidencia_feed.group(1)

    print("Feed extraído correctamente.")
    print("Longitud:", len(feed_resultados))

else:
    feed_resultados = None
    print("No fue posible extraer el feed.")

Feed extraído correctamente.
Longitud: 43772


In [41]:
print(feed_resultados[:3000])

SA÷1¬~ZA÷PANAMA: LPF - Clausura - Play Offs¬ZEE÷jHCVwi4L¬ZB÷149¬ZY÷Panama¬ZC÷K0V27L0b¬ZD÷p¬ZE÷MTDGpHcO¬ZF÷0¬ZO÷0¬ZG÷2¬ZH÷149_jHCVwi4L¬ZJ÷37¬ZL÷/panama/lpf/¬OAJ÷AsEohyiD-OEU1fTIL.png¬ZX÷00Panama      006Panama0000000000001000LPF - Clausu026y Offs000¬ZCC÷0¬TSS÷1¬ZAF÷Panama¬~AA÷zLN36mXI¬AD÷1526778000¬ADE÷1526778000¬AB÷3¬CR÷3¬AC÷3¬CX÷Independiente¬ER÷Final¬RW÷0¬AX÷0¬AO÷1526785326¬BW÷0¬BX÷-1¬WL÷¬WM÷CAI¬PX÷hjzxSMAK¬AE÷Independiente¬FH÷Independiente¬JA÷2iou0jYr¬WU÷independiente¬AS÷1¬AZ÷1¬BY÷2¬AJ÷1¬GRA÷0¬AG÷1¬AT÷1¬BC÷0¬OA÷IFMuGvXH-nXJKIqpI.png¬WN÷TAU¬PY÷bqtQiJd1¬AF÷Tauro¬FK÷Tauro¬JB÷pGzpaAmk¬WV÷tauro¬GRB÷0¬AH÷0¬AU÷0¬BD÷0¬OB÷S4FPavle-vTLj5hDm.png¬~AA÷YDObbJQi¬AD÷1526173200¬ADE÷1526173200¬AB÷3¬CR÷3¬AC÷3¬CX÷Independiente¬ER÷Semi-finals¬RW÷0¬AX÷0¬AO÷1526182014¬BW÷0¬BX÷-1¬WL÷¬WM÷CAI¬PX÷hjzxSMAK¬AE÷Independiente¬FH÷Independiente¬JA÷IT46bGCA¬WU÷independiente¬AS÷1¬AZ÷1¬BY÷1¬GRA÷0¬AG÷3¬AT÷3¬BC÷3¬OA÷IFMuGvXH-nXJKIqpI.png¬WN÷FRA¬PY÷COo0iase¬AF÷San Francisco¬FK÷San Francisco¬JB÷U38AczSG¬WV÷san-francisco¬G

### Celda 15.6. Inspección de la estructura interna del feed
Descompongo la cadena de datos obtenida dividiéndola por el delimitador `~` para analizar los bloques de información individual y comprender cómo Soccerway estructura cada campo (identificadores, fechas, equipos y marcadores).

In [42]:
from collections import Counter

# separación del feed en bloques individuales
bloques_feed = feed_resultados.split("~")

print("Cantidad de bloques:", len(bloques_feed))

Cantidad de bloques: 115


In [43]:
prefijos = []

for bloque in bloques_feed:
    bloque = bloque.strip()

    if "÷" in bloque:
        prefijo = bloque.split("÷", 1)[0]
        prefijos.append(prefijo)

conteo_prefijos = Counter(prefijos)

print("Prefijos detectados:")
for prefijo, cantidad in conteo_prefijos.most_common():
    print(f"{prefijo}: {cantidad}")

Prefijos detectados:
AA: 109
ZA: 4
SA: 1


In [44]:
print("\nPrimeros 40 bloques del feed:\n")

for i, bloque in enumerate(bloques_feed[:40]):
    print(i, "->", bloque[:300])


Primeros 40 bloques del feed:

0 -> SA÷1¬
1 -> ZA÷PANAMA: LPF - Clausura - Play Offs¬ZEE÷jHCVwi4L¬ZB÷149¬ZY÷Panama¬ZC÷K0V27L0b¬ZD÷p¬ZE÷MTDGpHcO¬ZF÷0¬ZO÷0¬ZG÷2¬ZH÷149_jHCVwi4L¬ZJ÷37¬ZL÷/panama/lpf/¬OAJ÷AsEohyiD-OEU1fTIL.png¬ZX÷00Panama      006Panama0000000000001000LPF - Clausu026y Offs000¬ZCC÷0¬TSS÷1¬ZAF÷Panama¬
2 -> AA÷zLN36mXI¬AD÷1526778000¬ADE÷1526778000¬AB÷3¬CR÷3¬AC÷3¬CX÷Independiente¬ER÷Final¬RW÷0¬AX÷0¬AO÷1526785326¬BW÷0¬BX÷-1¬WL÷¬WM÷CAI¬PX÷hjzxSMAK¬AE÷Independiente¬FH÷Independiente¬JA÷2iou0jYr¬WU÷independiente¬AS÷1¬AZ÷1¬BY÷2¬AJ÷1¬GRA÷0¬AG÷1¬AT÷1¬BC÷0¬OA÷IFMuGvXH-nXJKIqpI.png¬WN÷TAU¬PY÷bqtQiJd1¬AF÷Tauro¬FK÷
3 -> AA÷YDObbJQi¬AD÷1526173200¬ADE÷1526173200¬AB÷3¬CR÷3¬AC÷3¬CX÷Independiente¬ER÷Semi-finals¬RW÷0¬AX÷0¬AO÷1526182014¬BW÷0¬BX÷-1¬WL÷¬WM÷CAI¬PX÷hjzxSMAK¬AE÷Independiente¬FH÷Independiente¬JA÷IT46bGCA¬WU÷independiente¬AS÷1¬AZ÷1¬BY÷1¬GRA÷0¬AG÷3¬AT÷3¬BC÷3¬OA÷IFMuGvXH-nXJKIqpI.png¬WN÷FRA¬PY÷COo0iase¬AF÷San Fran
4 -> AA÷tvO2cwtb¬AD÷1526090400¬ADE÷1526090400¬AB÷3¬CR÷3¬AC÷10¬CX÷Arabe Un

### Celda 15.7. Conversión del feed de Soccerway a estructura tabular
Transformo los bloques estructurados obtenidos desde Soccerway en un DataFrame, conservando la fase competitive, identificador original, fecha, equipos participantes y marcador final. En esta etapa mantengo las denominaciones originales proporcionadas por la fuente para permitir su validación antes de aplicar la homologación con los registros de FootyStats.

In [45]:
# funcion para convertir cada bloque del feed en un diccionario
def parsear_bloque_soccerway(bloque):
    campos = {}

    for elemento in bloque.strip().split("¬"):
        if "÷" in elemento:
            clave, valor = elemento.split("÷", 1)
            campos[clave] = valor

    return campos


registros_soccerway = []
competicion_actual = None

for bloque in bloques_feed:

    bloque = bloque.strip()

    if not bloque:
        continue

    campos = parsear_bloque_soccerway(bloque)

    # los bloques ZA indican una nueva fase o competicion
    if bloque.startswith("ZA÷"):
        competicion_actual = campos.get("ZA")
        continue

    # los bloques AA corresponden a encuentros
    if bloque.startswith("AA÷"):

        registros_soccerway.append({
            "source_match_id": campos.get("AA"),
            "competicion_fuente": competicion_actual,
            "fase_fuente": campos.get("ER"),
            "timestamp": campos.get("AD"),
            "equipo_local_fuente": campos.get("AE"),
            "equipo_visitante_fuente": campos.get("AF"),
            "goles_local": campos.get("AG"),
            "goles_visitante": campos.get("AH"),
            "status_code": campos.get("AB")
        })


df_soccerway_raw = pd.DataFrame(registros_soccerway)

print("Encuentros extraídos:", len(df_soccerway_raw))
print("Columnas generadas:", df_soccerway_raw.shape[1])

display(df_soccerway_raw.head(10))

Encuentros extraídos: 109
Columnas generadas: 9


,source_match_id,competicion_fuente,fase_fuente,timestamp,equipo_local_fuente,equipo_visitante_fuente,goles_local,goles_visitante,status_code
0,zLN36mXI,PANAMA: LPF - Clausura - Play Offs,Final,1526778000,Independiente,Tauro,1,0,3
1,YDObbJQi,PANAMA: LPF - Clausura - Play Offs,Semi-finals,1526173200,Independiente,San Francisco,3,1,3
2,tvO2cwtb,PANAMA: LPF - Clausura - Play Offs,Semi-finals,1526090400,Arabe Unido,Tauro,2,2,3
3,fNPfaaBo,PANAMA: LPF - Clausura - Play Offs,Semi-finals,1525564800,Tauro,Arabe Unido,2,0,3
4,hIoCzGYp,PANAMA: LPF - Clausura - Play Offs,Semi-finals,1525485600,San Francisco,Independiente,1,0,3
5,vymYlbuU,PANAMA: LPF - Clausura,Round 18,1524956400,Alianza,Deportivo Universitario,0,1,3
6,pEmUkvQN,PANAMA: LPF - Clausura,Round 18,1524956400,Arabe Unido,San Francisco,1,1,3
7,AeeHhtt5,PANAMA: LPF - Clausura,Round 18,1524956400,Plaza Amador,Independiente,0,0,3
8,lnfLi0eB,PANAMA: LPF - Clausura,Round 18,1524956400,Tauro,Miguelito,3,0,3
9,6NlQjKAH,PANAMA: LPF - Clausura,Round 18,1524879000,Santa Gema,Atl. Veraguense,1,0,3


### Celda 15.8. Normalización inicial de tipos de datos
Convierto el timestamp y las variables de marcador a tipos adecuados para facilitar las comprobaciones posteriores. En esta etapa todavía no modifico los nombres de los equipos ni integro los registros con FootyStats.

In [46]:
# timestamp numerico
df_soccerway_raw["timestamp"] = pd.to_numeric(
    df_soccerway_raw["timestamp"],
    errors="coerce"
)

# conversion temporal
df_soccerway_raw["fecha_partido"] = pd.to_datetime(
    df_soccerway_raw["timestamp"],
    unit="s",
    errors="coerce",
    utc=True
)

# conversion de goles
df_soccerway_raw["goles_local"] = pd.to_numeric(
    df_soccerway_raw["goles_local"],
    errors="coerce"
).astype("Int64")

df_soccerway_raw["goles_visitante"] = pd.to_numeric(
    df_soccerway_raw["goles_visitante"],
    errors="coerce"
).astype("Int64")

# metadatos de procedencia
df_soccerway_raw["temporada"] = "2017/2018"
df_soccerway_raw["fuente"] = "Soccerway"

display(
    df_soccerway_raw[
        [
            "fecha_partido",
            "competicion_fuente",
            "fase_fuente",
            "equipo_local_fuente",
            "equipo_visitante_fuente",
            "goles_local",
            "goles_visitante"
        ]
    ].head(10)
)

,fecha_partido,competicion_fuente,fase_fuente,equipo_local_fuente,equipo_visitante_fuente,goles_local,goles_visitante
0,2018-05-20 01:00:00+00:00,PANAMA: LPF - Clausura - Play Offs,Final,Independiente,Tauro,1,0
1,2018-05-13 01:00:00+00:00,PANAMA: LPF - Clausura - Play Offs,Semi-finals,Independiente,San Francisco,3,1
2,2018-05-12 02:00:00+00:00,PANAMA: LPF - Clausura - Play Offs,Semi-finals,Arabe Unido,Tauro,2,2
3,2018-05-06 00:00:00+00:00,PANAMA: LPF - Clausura - Play Offs,Semi-finals,Tauro,Arabe Unido,2,0
4,2018-05-05 02:00:00+00:00,PANAMA: LPF - Clausura - Play Offs,Semi-finals,San Francisco,Independiente,1,0
5,2018-04-28 23:00:00+00:00,PANAMA: LPF - Clausura,Round 18,Alianza,Deportivo Universitario,0,1
6,2018-04-28 23:00:00+00:00,PANAMA: LPF - Clausura,Round 18,Arabe Unido,San Francisco,1,1
7,2018-04-28 23:00:00+00:00,PANAMA: LPF - Clausura,Round 18,Plaza Amador,Independiente,0,0
8,2018-04-28 23:00:00+00:00,PANAMA: LPF - Clausura,Round 18,Tauro,Miguelito,3,0
9,2018-04-28 01:30:00+00:00,PANAMA: LPF - Clausura,Round 18,Santa Gema,Atl. Veraguense,1,0


### Celda 15.9. Auditoría de cobertura de los registros extraídos
Verifico la cantidad de encuentros recuperados por fase competitiva, su rango temporal y la existencia de valores faltantes. Esta revisión permite determinar si el feed obtenido representa la temporada 2017/2018 completa antes de proceder con su integración al conjunto histórico.

In [47]:
# conteo general de partidos recuperados
print("Total de encuentros:", len(df_soccerway_raw))

# verificacion de la ventana temporal
print("\nRango temporal:")
print("Fecha inicial:", df_soccerway_raw["fecha_partido"].min())
print("Fecha final:", df_soccerway_raw["fecha_partido"].max())

# desglose por competicion/fase
print("\nEncuentros por competición:")
display(
    df_soccerway_raw["competicion_fuente"]
    .value_counts(dropna=False)
)

# chequeo de integridad y valores nulos
print("\nValores faltantes:")
display(
    df_soccerway_raw[
        [
            "source_match_id",
            "fecha_partido",
            "competicion_fuente",
            "fase_fuente",
            "equipo_local_fuente",
            "equipo_visitante_fuente",
            "goles_local",
            "goles_visitante"
        ]
    ].isna().sum()
)

Total de encuentros: 109

Rango temporal:
Fecha inicial: 2017-10-28 23:00:00+00:00
Fecha final: 2018-05-20 01:00:00+00:00

Encuentros por competición:


competicion_fuente
PANAMA: LPF - Clausura                90
PANAMA: LPF - Apertura                 9
PANAMA: LPF - Clausura - Play Offs     5
PANAMA: LPF - Apertura - Play Offs     5
Name: count, dtype: int64


Valores faltantes:


source_match_id            0
fecha_partido              0
competicion_fuente         0
fase_fuente                0
equipo_local_fuente        0
equipo_visitante_fuente    0
goles_local                0
goles_visitante            0
dtype: int64

In [48]:
equipos_soccerway = sorted(
    set(
        df_soccerway_raw["equipo_local_fuente"]
        .dropna()
    ).union(
        set(
            df_soccerway_raw["equipo_visitante_fuente"]
            .dropna()
        )
    )
)

print(
    "Denominaciones únicas encontradas:",
    len(equipos_soccerway)
)

for equipo in equipos_soccerway:
    print("-", equipo)

Denominaciones únicas encontradas: 10
- Alianza
- Arabe Unido
- Atl. Veraguense
- Deportivo Universitario
- Independiente
- Miguelito
- Plaza Amador
- San Francisco
- Santa Gema
- Tauro


### Celda 15.11. Distribución de partidos por ronda y competición
Analizo el desglose de encuentros según la competición de origen (`competicion_fuente`) y la fase correspondiente (`fase_fuente`). Esta agrupación permite verificar el balance de partidos entre las distintas fases (por ejemplo, Apertura, Clausura o playoffs) dentro de los registros extraídos de Soccerway.

In [49]:
# conteo de partidos agrupados por torneo y jornada/fase
conteo_rondas = (
    df_soccerway_raw
    .groupby(
        ["competicion_fuente", "fase_fuente"],
        dropna=False
    )
    .size()
    .rename("partidos")
    .reset_index()
)

display(conteo_rondas)

,competicion_fuente,fase_fuente,partidos
0,PANAMA: LPF - Apertura,Round 17,5
1,PANAMA: LPF - Apertura,Round 18,4
2,PANAMA: LPF - Apertura - Play Offs,Final,1
3,PANAMA: LPF - Apertura - Play Offs,Semi-finals,4
4,PANAMA: LPF - Clausura,Round 1,5
5,PANAMA: LPF - Clausura,Round 10,5
6,PANAMA: LPF - Clausura,Round 11,5
7,PANAMA: LPF - Clausura,Round 12,5
8,PANAMA: LPF - Clausura,Round 13,5
9,PANAMA: LPF - Clausura,Round 14,5


### Celda 15.12. Verificación de cobertura de la temporada 2017/2018
Comparo la cantidad de encuentros obtenidos mediante Soccerway con la estructura histórica esperada de la temporada. Esta comprobación permite identificar posibles limitaciones en la carga inicial de la página (por ejemplo, paginación o la necesidad de peticiones AJAX adicionales) antes de incorporar los registros al conjunto histórico principal.

In [50]:
# tabla comparativa entre registros esperados y extraidos
auditoria_cobertura = pd.DataFrame({
    "componente": [
        "Apertura - Fase regular",
        "Apertura - Playoffs",
        "Clausura - Fase regular",
        "Clausura - Playoffs"
    ],
    "esperados": [90, 5, 90, 5],
    "extraidos": [9, 5, 90, 5]
})

# calculo de faltantes por componente
auditoria_cobertura["faltantes"] = (
    auditoria_cobertura["esperados"]
    - auditoria_cobertura["extraidos"]
)

display(auditoria_cobertura)

# resumen global de cobertura
print(
    "\nTotal esperado:",
    auditoria_cobertura["esperados"].sum()
)

print(
    "Total extraído:",
    auditoria_cobertura["extraidos"].sum()
)

print(
    "Total faltante:",
    auditoria_cobertura["faltantes"].sum()
)

,componente,esperados,extraidos,faltantes
0,Apertura - Fase regular,90,9,81
1,Apertura - Playoffs,5,5,0
2,Clausura - Fase regular,90,90,0
3,Clausura - Playoffs,5,5,0



Total esperado: 190
Total extraído: 109
Total faltante: 81


### Celda 15.13 Extracción completa de la temporada mediante automatización web (Selenium)
Dado que la carga estática inicial del HTML solo incluye un subconjunto limitado de partidos (debido a la paginación o carga dinámica vía AJAX/JavaScript en Soccerway), utilizo **Selenium** para interactuar con la página web, simular la navegación y cargar la totalidad de los encuentros correspondientes a los torneos Apertura y Clausura de la temporada 2017/2018.

In [51]:
%pip install selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# configuracion del navegador automatizado
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # ejecucion en segundo plano
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/154.0.0.0 Safari/537.36"
)

# inicializacion del driver
driver = webdriver.Chrome(options=options)

print("Accediendo a la página de Soccerway...")
driver.get("https://www.soccerway.com/panama/lpf-2017-2018/results/")

# espera explicita para la carga inicial del DOM
wait = WebDriverWait(driver, 15)
wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
time.sleep(3)

# busqueda de partidos renderizados con la sesion activa
partidos_renderizados = driver.find_elements(
    By.CSS_SELECTOR,
    ".event__match"
)

print(
    "Partidos actualmente renderizados:",
    len(partidos_renderizados)
)

Accediendo a la página de Soccerway...
Partidos actualmente renderizados: 109


### Celda 15.14 Verificación de partidos renderizados en el DOM
Inspecciono el número de elementos de clase `.event__match` presentes en el documento HTML renderizado dinámicamente. Esta verificación permite evaluar si la sesión de automatización con Selenium cargó adecuadamente la estructura del DOM antes de aplicar la extracción de los registros.

In [53]:
# busqueda de elementos de partidos renderizados en el DOM
partidos_renderizados = driver.find_elements(
    By.CSS_SELECTOR,
    ".event__match"
)

print(
    "Partidos actualmente renderizados:",
    len(partidos_renderizados)
)

Partidos actualmente renderizados: 109


### Celda 15.15  Carga progresiva de resultados históricos
Debido a que Soccerway carga inicialmente solo una parte de los encuentros históricos, utilizo Selenium para activar de manera progresiva la opción de mostrar más partidos. Después de cada interacción verifico el incremento en la cantidad de encuentros renderizados, evitando continuar cuando no existan nuevos registros o cuando se alcance la cobertura esperada de la temporada.

In [54]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException
)

URL_SOCCERWAY = (
    "https://www.soccerway.com/panama/"
    "lpf-2017-2018/results/"
)

options = webdriver.ChromeOptions()
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
driver.get(URL_SOCCERWAY)

wait = WebDriverWait(driver, 15)

print("Página cargada:", driver.title)

# Intentar aceptar cookies si aparece el cuadro
selectores_cookies = [
    "button#onetrust-accept-btn-handler",
    "button[aria-label*='Accept']",
    "button[aria-label*='Aceptar']"
]

for selector in selectores_cookies:
    try:
        boton_cookie = WebDriverWait(driver, 3).until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, selector)
            )
        )
        boton_cookie.click()
        print("Aviso de cookies cerrado.")
        break
    except Exception:
        pass

time.sleep(3)

def contar_partidos(driver):
    return len(
        driver.find_elements(
            By.CSS_SELECTOR,
            ".event__match"
        )
    )

print(
    "Partidos iniciales:",
    contar_partidos(driver)
)

Página cargada: Soccerway
Aviso de cookies cerrado.
Partidos iniciales: 109


### Celda 15.16  Inspección y filtrado de los botones de carga por XPath
Con el objetivo de garantizar la localización precisa del botón "Show more matches" cuando los selectores CSS genéricos no devuelven resultados directos, utilizo una búsqueda basada en la expresión XPath `//span[normalize-space()='Show more matches']`. Filtro únicamente aquellos elementos que se encuentran visibles en el DOM para poder inspeccionar sus propiedades e interactuar con ellos en la siguiente etapa.

In [57]:
from selenium.webdriver.common.by import By

# localizacion explicita mediante XPath basado en texto exacto
show_more = driver.find_elements(
    By.XPATH,
    "//span[normalize-space()='Show more matches']"
)

# filtrado de elementos unicamente visibles en pantalla
show_more_visibles = [
    elemento for elemento in show_more
    if elemento.is_displayed()
]

print("Elementos visibles 'Show more matches':", len(show_more_visibles))

# impresion de metadatos de los elementos localizados
for i, elemento in enumerate(show_more_visibles):
    print(
        f"{i}:",
        "tag =", elemento.tag_name,
        "| class =", elemento.get_attribute("class")
    )

Elementos visibles 'Show more matches': 1
0: tag = span | class = wcl-bold_NZXv6 wcl-caption-05_SUCBa wcl-scores_Na715


In [58]:
if show_more_visibles:

    elemento = show_more_visibles[0]

    actual = elemento

    for nivel in range(5):

        print(f"\nNIVEL {nivel}")
        print("Tag:", actual.tag_name)
        print("Class:", actual.get_attribute("class"))
        print("Role:", actual.get_attribute("role"))

        html = actual.get_attribute("outerHTML")

        print(html[:1500])

        try:
            actual = actual.find_element(By.XPATH, "..")
        except:
            break


NIVEL 0
Tag: span
Class: wcl-bold_NZXv6 wcl-caption-05_SUCBa wcl-scores_Na715
Role: None
<span class="wcl-bold_NZXv6 wcl-caption-05_SUCBa wcl-scores_Na715" data-testid="wcl-caption-05">Show more matches</span>

NIVEL 1
Tag: button
Class: wcl-footer__button_OauhJ wcl-buttonLink_jmSkY wcl-primary_aIST5 wcl-underline_rL72U
Role: None
<button class="wcl-footer__button_OauhJ wcl-buttonLink_jmSkY wcl-primary_aIST5 wcl-underline_rL72U" data-testid="wcl-buttonLink"><span class="wcl-bold_NZXv6 wcl-caption-05_SUCBa wcl-scores_Na715" data-testid="wcl-caption-05">Show more matches</span><svg fill="currentColor" viewBox="0 0 20 20" data-testid="wcl-icon-action-navigation-arrow-down" class="wcl-icon_WGKvC" width="12" height="12" aria-hidden="true"><path fill-rule="evenodd" class="action-navigation-arrow-down" d="M19.02 5 10 14.03.98 5 0 5.98l10 10 10-10-.98-.98Z"></path></svg></button>

NIVEL 2
Tag: div
Class: wcl-footer_yI6S3
Role: None
<div class="wcl-footer_yI6S3" data-testid="wcl-section-footer

In [59]:
if show_more_visibles:

    boton_texto = show_more_visibles[0]

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        boton_texto
    )

    time.sleep(1)

    partidos_antes = contar_partidos(driver)

    driver.execute_script(
        "arguments[0].click();",
        boton_texto
    )

    time.sleep(5)

    partidos_despues = contar_partidos(driver)

    print("Partidos antes:", partidos_antes)
    print("Partidos después:", partidos_despues)

Partidos antes: 109
Partidos después: 190


In [60]:
max_intentos = 10

for intento in range(1, max_intentos + 1):

    partidos_antes = contar_partidos(driver)

    elementos = driver.find_elements(
        By.XPATH,
        "//span[normalize-space()='Show more matches']"
    )

    visibles = [
        elemento for elemento in elementos
        if elemento.is_displayed()
    ]

    print(f"\nIntento {intento}")
    print("Partidos actuales:", partidos_antes)

    if not visibles:
        print("No quedan más resultados para cargar.")
        break

    boton = visibles[0]

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        boton
    )

    time.sleep(1)

    driver.execute_script(
        "arguments[0].click();",
        boton
    )

    # Esperar hasta que aumente el número de encuentros
    limite = time.time() + 15

    partidos_despues = partidos_antes

    while time.time() < limite:

        partidos_despues = contar_partidos(driver)

        if partidos_despues > partidos_antes:
            break

        time.sleep(1)

    print(
        f"Partidos: {partidos_antes} → {partidos_despues}"
    )

    if partidos_despues >= 190:
        print("Cobertura esperada alcanzada.")
        break

    if partidos_despues == partidos_antes:
        print("El clic no produjo nuevos encuentros.")
        break


Intento 1
Partidos actuales: 190
No quedan más resultados para cargar.


### Celda 15.17 Definición de rutas y respaldo en disco del HTML renderizado
Defino la estructura de directorios del proyecto utilizando la librería `pathlib` para asegurar la existencia de la carpeta de almacenamiento de datos crudos (`data/raw/soccerway`).Extraigo el documento HTML completamente renderizado desde la sesión activa de Selenium (con los 190 partidos cargados) y lo guardo localmente como respaldo, permitiendo continuar el procesamiento de datos de forma offline sin depender de nuevas peticiones web.

In [62]:
from pathlib import Path

# definicion de la raiz del proyecto y directorio de almacenamiento
ROOT = Path("..").resolve()
SOCCERWAY_DIR = ROOT / "data" / "raw" / "soccerway"

# creacion del directorio si no existe previamente
SOCCERWAY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# captura del codigo fuente HTML renderizado con todos los partidos cargados
html_completo = driver.page_source

HTML_COMPLETO = (
    SOCCERWAY_DIR
    / "lpf_2017_2018_soccerway_completo.html"
)

# guardado del documento HTML en disco con codificacion UTF-8
HTML_COMPLETO.write_text(
    html_completo,
    encoding="utf-8"
)

# verificacion del archivo guardado y metadatos de espacio
print("HTML completo guardado en:")
print(HTML_COMPLETO)

print(
    "Tamaño:",
    HTML_COMPLETO.stat().st_size,
    "bytes"
)

print(
    "Partidos renderizados:",
    contar_partidos(driver)
)

HTML completo guardado en:
C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\raw\soccerway\lpf_2017_2018_soccerway_completo.html
Tamaño: 1581345 bytes
Partidos renderizados: 190


### Celda 15.18. Inspección de la estructura de un encuentro renderizado
Inspecciono la estructura HTML de los encuentros cargados dinámicamente para identificar los selectores correspondientes a fecha, equipos, marcador e identificador del partido antes de realizar la extracción definitiva.

In [63]:
from selenium.webdriver.common.by import By

# localizacion de los elementos de partidos en el DOM renderizado
partidos_dom = driver.find_elements(
    By.CSS_SELECTOR,
    ".event__match"
)

print("Partidos detectados:", len(partidos_dom))

# impresion del HTML de los primeros dos encuentros para verificar selectores
for i, partido in enumerate(partidos_dom[:2]):

    print("\n" + "=" * 120)
    print(f"PARTIDO {i}")
    print("=" * 120)

    html_partido = partido.get_attribute(
        "outerHTML"
    )

    print(html_partido[:6000])

Partidos detectados: 190

PARTIDO 0
<div id="g_1_zLN36mXI" data-sport="soccer" data-is-duel="true" data-is-doubles="false" data-has-moneyline="false" class="event__match event__match--withRowLink event__match--twoLine" data-event-row="true"><a href="https://www.soccerway.com/match/independiente-hjzxSMAK/tauro-bqtQiJd1/?mid=zLN36mXI" target="_self" class="eventRowLink" title="Click for details!" aria-label="Click for details!" id="match-row-g_1_zLN36mXI"></a><div class="eventStarTouchZone"></div><span class="wcl-simple-text-01_EaZ9- wcl-scores_Na715 wcl-stageTime_TQm5m event__stageTime event__stageTime--date" data-testid="wcl-stageTime" data-live="false" data-blinking="false" data-highlighted="false" data-linethrough="false"><span class="wcl-simple-text-01_EaZ9- wcl-scores_Na715 wcl-dateContent_eEChT" data-testid="wcl-simple-text-01">19.05.2018</span></span><div class="wcl-participant_bctDY event__homeParticipant" data-testid="wcl-matchRow-participant"><div class="wcl-participants_ASufu

### Celda 15.19 Revisión de atributos de metadatos del elemento
Analizo los atributos HTML del primer partido detectado (`id`, `data-start-time` y `data-event-id`) para verificar si Soccerway almacena marcas temporales en formato Unix timestamp o identificadores únicos dentro del DOM renderizado. Esta información permitirá mapear las fechas exactas de los encuentros y vincularlos con el resto del pipeline de datos.

In [65]:
# asignacion del primer elemento del DOM para la inspeccion de atributos
primer_partido = partidos_dom[0]

# extraccion de atributos clave de tiempo e identificadores
print(
    "ID del elemento:",
    primer_partido.get_attribute("id")
)

print(
    "data-start-time:",
    primer_partido.get_attribute(
        "data-start-time"
    )
)

print(
    "data-event-id:",
    primer_partido.get_attribute(
        "data-event-id"
    )
)

ID del elemento: g_1_zLN36mXI
data-start-time: None
data-event-id: None


### Celda 15.20. Extracción definitiva de los encuentros 2017/2018
A partir del documento completamente renderizado mediante la sesión activa de Selenium, extraigo los 190 encuentros correspondientes a la temporada 2017/2018. Para cada registro recupero el identificador único de Soccerway (`source_match_id`), la fecha local, los equipos participantes y el marcador final. Las denominaciones originales de los clubes se conservan para realizar posteriormente su homologación con la nomenclatura utilizada en FootyStats.

In [66]:
import re
import pandas as pd
from selenium.webdriver.common.by import By

# localizacion de los elementos de partidos renderizados en el DOM
partidos_dom = driver.find_elements(
    By.CSS_SELECTOR,
    ".event__match"
)

registros_2017_2018 = []

for partido in partidos_dom:

    # ID original de Soccerway
    id_elemento = partido.get_attribute("id")

    if id_elemento:
        partes_id = id_elemento.split("_", 2)

        if len(partes_id) == 3:
            source_match_id = partes_id[2]
        else:
            source_match_id = id_elemento
    else:
        source_match_id = None

    # ---------------------------
    # Fecha
    # ---------------------------
    fecha = partido.find_element(
        By.CSS_SELECTOR,
        ".event__stageTime--date"
    ).text.strip()

    # Equipo local
    local = partido.find_element(
        By.CSS_SELECTOR,
        ".event__homeParticipant"
    )

    local = local.find_element(
        By.CSS_SELECTOR,
        "[data-testid='wcl-simple-text-01']"
    ).text.strip()

    # Equipo visitante
    visitante = partido.find_element(
        By.CSS_SELECTOR,
        ".event__awayParticipant"
    )

    visitante = visitante.find_element(
        By.CSS_SELECTOR,
        "[data-testid='wcl-simple-text-01']"
    ).text.strip()

    # Marcador
    goles_local = partido.find_element(
        By.CSS_SELECTOR,
        ".event__score--home"
    ).text.strip()

    goles_visitante = partido.find_element(
        By.CSS_SELECTOR,
        ".event__score--away"
    ).text.strip()

    registros_2017_2018.append({
        "source_match_id": source_match_id,
        "fecha_fuente": fecha,
        "equipo_local_fuente": local,
        "equipo_visitante_fuente": visitante,
        "goles_local": goles_local,
        "goles_visitante": goles_visitante
    })


df_soccerway_completo = pd.DataFrame(
    registros_2017_2018
)

print(
    "Partidos extraídos:",
    len(df_soccerway_completo)
)

display(
    df_soccerway_completo.head(10)
)

Partidos extraídos: 190


,source_match_id,fecha_fuente,equipo_local_fuente,equipo_visitante_fuente,goles_local,goles_visitante
0,zLN36mXI,19.05.2018,Independiente,Tauro,1,0
1,YDObbJQi,12.05.2018,Independiente,San Francisco,3,1
2,tvO2cwtb,11.05.2018\nAET,Arabe Unido,Tauro,2,2
3,fNPfaaBo,05.05.2018,Tauro,Arabe Unido,2,0
4,hIoCzGYp,04.05.2018,San Francisco,Independiente,1,0
5,vymYlbuU,28.04.2018,Alianza,Deportivo Universitario,0,1
6,pEmUkvQN,28.04.2018,Arabe Unido,San Francisco,1,1
7,AeeHhtt5,28.04.2018,Plaza Amador,Independiente,0,0
8,lnfLi0eB,28.04.2018,Tauro,Miguelito,3,0
9,6NlQjKAH,27.04.2018,Santa Gema,Atl. Veraguense,1,0


### Celda 15.21. Normalización de fechas y marcadores
Convierto las fechas y los marcadores recuperados a tipos de datos adecuados para su análisis (`datetime64[ns]` e `Int64`). La fecha proporcionada por la interfaz corresponde al formato de fecha local mostrado por Soccerway (`DD.MM.YYYY`), por lo que se transforma para establecer la referencia cronológica precisa de la temporada 2017/2018.

In [67]:
# Fecha local del partido
df_soccerway_completo["fecha_partido"] = pd.to_datetime(
    df_soccerway_completo["fecha_fuente"],
    format="%d.%m.%Y",
    errors="coerce"
)

# Marcadores
df_soccerway_completo["goles_local"] = pd.to_numeric(
    df_soccerway_completo["goles_local"],
    errors="coerce"
).astype("Int64")

df_soccerway_completo["goles_visitante"] = pd.to_numeric(
    df_soccerway_completo["goles_visitante"],
    errors="coerce"
).astype("Int64")

# Metadatos
df_soccerway_completo["temporada"] = "2017/2018"
df_soccerway_completo["fuente"] = "Soccerway"

print(
    "Fechas inválidas:",
    df_soccerway_completo["fecha_partido"].isna().sum()
)

print(
    "Marcadores locales inválidos:",
    df_soccerway_completo["goles_local"].isna().sum()
)

print(
    "Marcadores visitantes inválidos:",
    df_soccerway_completo["goles_visitante"].isna().sum()
)

print(
    "\nRango temporal:"
)

print(
    "Fecha inicial:",
    df_soccerway_completo["fecha_partido"].min()
)

print(
    "Fecha final:",
    df_soccerway_completo["fecha_partido"].max()
)

Fechas inválidas: 3
Marcadores locales inválidos: 1
Marcadores visitantes inválidos: 1

Rango temporal:
Fecha inicial: 2017-07-14 00:00:00
Fecha final: 2018-05-19 00:00:00


### Celda 15.22 Auditoría de registros duplicados
Realizo una comprobación de integridad en `df_soccerway_completo` para detectar posibles duplicaciones resultantes del proceso de carga progresiva o renderizado en el DOM. Evalúo tanto la presencia de identificadores idénticos (`source_match_id`) como la duplicidad funcional combinando fecha, equipo local y equipo visitante.

In [68]:
# Verificacion de IDs de partido duplicados en el conjunto extraido
print(
    "IDs duplicados:",
    df_soccerway_completo[
        "source_match_id"
    ].duplicated().sum()
)

# Identificacion de duplicados basados en la tupla (fecha, local, visitante)
duplicados_partido = (
    df_soccerway_completo
    .duplicated(
        subset=[
            "fecha_partido",
            "equipo_local_fuente",
            "equipo_visitante_fuente"
        ],
        keep=False
    )
)

print(
    "Registros duplicados por fecha + local + visitante:",
    duplicados_partido.sum()
)

# Despliegue de los registros duplicados en caso de existencia
if duplicados_partido.any():

    display(
        df_soccerway_completo.loc[
            duplicados_partido,
            [
                "source_match_id",
                "fecha_partido",
                "equipo_local_fuente",
                "equipo_visitante_fuente",
                "goles_local",
                "goles_visitante"
            ]
        ]
    )

IDs duplicados: 0
Registros duplicados por fecha + local + visitante: 0


### Celda 15.23 Verificación de los clubes participantes
Consolido los nombres únicos de los equipos presentes en las columnas de local y visitante de `df_soccerway_completo`. Esta inspección confirma la presencia de los 10 clubes de la LPF para la temporada 2017/2018 y permite revisar sus denominaciones exactas para definir posteriormente la tabla de homologación con FootyStats.

In [69]:
# obtencion y consolidacion de nombres unicos de equipos
equipos_soccerway_completo = sorted(
    set(
        df_soccerway_completo[
            "equipo_local_fuente"
        ].dropna()
    ).union(
        set(
            df_soccerway_completo[
                "equipo_visitante_fuente"
            ].dropna()
        )
    )
)

print(
    "Total de denominaciones:",
    len(equipos_soccerway_completo)
)

# listado de clubes detectados en el dataset
for equipo in equipos_soccerway_completo:
    print("-", equipo)

Total de denominaciones: 10
- Alianza
- Arabe Unido
- Atl. Veraguense
- Deportivo Universitario
- Independiente
- Miguelito
- Plaza Amador
- San Francisco
- Santa Gema
- Tauro


### Celda 15.24 Verificación matemática de participaciones por equipo
Calculo la frecuencia total de partidos disputados por cada club sumando sus apariciones como local (`equipo_local_fuente`) y como visitante (`equipo_visitante_fuente`). Esta auditoría confirma la integridad del conteo global, asegurando que la suma de participaciones individuales equivalga exactamente al doble del número total de encuentros registrados ($190 \times 2 = 380$).

In [70]:
# Conteo de partidos como local y visitante por club
apariciones_local_1718 = (
    df_soccerway_completo[
        "equipo_local_fuente"
    ]
    .value_counts()
)

apariciones_visitante_1718 = (
    df_soccerway_completo[
        "equipo_visitante_fuente"
    ]
    .value_counts()
)

# Consolidacion de participaciones totales por equipo
frecuencia_1718 = (
    apariciones_local_1718
    .add(
        apariciones_visitante_1718,
        fill_value=0
    )
    .astype(int)
    .sort_values(ascending=False)
)

display(
    frecuencia_1718
    .rename("partidos")
    .to_frame()
)

# Verificacion de consistencia matematica global
print(
    "\nApariciones observadas:",
    frecuencia_1718.sum()
)

print(
    "Apariciones esperadas:",
    len(df_soccerway_completo) * 2
)

print(
    "Consistencia:",
    frecuencia_1718.sum()
    == len(df_soccerway_completo) * 2
)

,partidos
equipo_local_fuente,
Arabe Unido,41
Tauro,41
Independiente,39
Deportivo Universitario,39
San Francisco,38
Plaza Amador,38
Alianza,36
Atl. Veraguense,36
Miguelito,36



Apariciones observadas: 380
Apariciones esperadas: 380
Consistencia: True


### Celda 15.25. Homologación de las denominaciones de los equipos
Las denominaciones utilizadas por Soccerway presentan diferencias respecto a los nombres empleados en los archivos históricos de FootyStats. Para permitir la integración fluida de ambas fuentes en el dataset global, defino el diccionario de mapeo `MAPEO_EQUIPOS_1718` y aplico la transformación sobre los campos de equipo local y visitante, respetando las identidades institucionales históricas de la temporada 2017/2018 (como la denominación previa de Chorrillo FC antes de su transición a CD Universitario).

In [ ]:
# diccionario de mapeo de nombres Soccerway -> FootyStats (2017/2018)
MAPEO_EQUIPOS_1718 = {
    "Alianza": "Alianza",
    "Arabe Unido": "Árabe Unido",
    "Atl. Veraguense": "Atlético Veragüense",
    "Deportivo Universitario": "Chorrillo",
    "Independiente": "CA Independiente de La Chorrera",
    "Miguelito": "Sporting San Miguelito",
    "Plaza Amador": "Plaza Amador",
    "San Francisco": "San Francisco FC",
    "Santa Gema": "Santa Gema",
    "Tauro": "Tauro"
}

# aplicacion del mapeo a las columnas de equipos
df_soccerway_completo["equipo_local"] = (
    df_soccerway_completo["equipo_local_fuente"]
    .map(MAPEO_EQUIPOS_1718)
)

df_soccerway_completo["equipo_visitante"] = (
    df_soccerway_completo["equipo_visitante_fuente"]
    .map(MAPEO_EQUIPOS_1718)
)

# verificacion de valores no mapeados (debe dar 0)
print(
    "Locales sin mapear:",
    df_soccerway_completo["equipo_local"].isna().sum()
)

print(
    "Visitantes sin mapear:",
    df_soccerway_completo["equipo_visitante"].isna().sum()
)

display(
    df_soccerway_completo[
        [
            "fecha_partido",
            "equipo_local_fuente",
            "equipo_local",
            "equipo_visitante_fuente",
            "equipo_visitante",
            "goles_local",
            "goles_visitante"
        ]
    ].head(10)
)

### Celda 15.26. Identificación precisa de anomalías de parseo y conversión
Mascaro los registros de `df_soccerway_completo` para filtrar cualquier fila que presente valores nulos (`NaN` / `NA`) en la fecha convertida o en los marcadores de goles. Esta auditoría permite aislar inconsistencias de formato originadas en el scrapeo (como partidos suspendidos, aplazados o cadenas de texto no numéricas) antes de realizar la exportación o consolidación final.

In [72]:
# detectar fechas que no pudieron convertirse
mask_fecha_invalida = (
    df_soccerway_completo["fecha_partido"].isna()
)

# detectar marcadores que no pudieron convertirse
mask_gol_local_invalido = (
    df_soccerway_completo["goles_local"].isna()
)

mask_gol_visitante_invalido = (
    df_soccerway_completo["goles_visitante"].isna()
)

# consolidacion de mascaras de anomalias
mask_anomalias = (
    mask_fecha_invalida
    | mask_gol_local_invalido
    | mask_gol_visitante_invalido
)

columnas_revision = [
    "source_match_id",
    "fecha_fuente",
    "equipo_local_fuente",
    "equipo_visitante_fuente",
    "goles_local",
    "goles_visitante"
]

# despliegue de registros anomalos detectados
display(
    df_soccerway_completo.loc[
        mask_anomalias,
        columnas_revision
    ]
)

,source_match_id,fecha_fuente,equipo_local_fuente,equipo_visitante_fuente,goles_local,goles_visitante
2,tvO2cwtb,11.05.2018\nAET,Arabe Unido,Tauro,2,2
70,YeopAmAn,03.02.2018\nAwrd,Alianza,Tauro,3,0
97,j36Txm26,25.11.2017\nPen,Plaza Amador,Arabe Unido,<NA>,<NA>


In [73]:
df_revision_raw = pd.DataFrame(
    registros_2017_2018
)

display(
    df_revision_raw.loc[
        df_revision_raw["source_match_id"].isin(
            df_soccerway_completo.loc[
                mask_anomalias,
                "source_match_id"
            ]
        )
    ]
)

,source_match_id,fecha_fuente,equipo_local_fuente,equipo_visitante_fuente,goles_local,goles_visitante
2,tvO2cwtb,11.05.2018\nAET,Arabe Unido,Tauro,2,2
70,YeopAmAn,03.02.2018\nAwrd,Alianza,Tauro,3,0
97,j36Txm26,25.11.2017\nPen,Plaza Amador,Arabe Unido,0\n(4),1\n(5)


### Celda 15.27. Limpieza de fechas y extracción de estados especiales de los encuentros
Dado que el texto del elemento de fecha extraído desde Soccerway (`.event__stageTime--date`) puede incluir saltos de línea con marcadores de estado especial (como partidos suspendidos, pospuestos o en prórroga), utilizo expresiones regulares para separar la cadena en dos componentes: la fecha limpia en formato `DD.MM.YYYY` y la etiqueta de estado correspondiente (`estado_especial`). Finalmente, convierto la fecha a `datetime64[ns]` y audito los valores resultantes.

In [74]:
import pandas as pd

# conservar evidencia original de la cadena extraida
df_soccerway_completo["fecha_fuente_original"] = (
    df_soccerway_completo["fecha_fuente"]
    .astype("string")
)

# extraer solamente la subcadena de fecha dd.mm.aaaa
df_soccerway_completo["fecha_limpia"] = (
    df_soccerway_completo["fecha_fuente_original"]
    .str.extract(
        r"(\d{2}\.\d{2}\.\d{4})",
        expand=False
    )
)

# identificar estado especial mostrado por Soccerway despus del salto de linea
df_soccerway_completo["estado_especial"] = (
    df_soccerway_completo["fecha_fuente_original"]
    .str.extract(
        r"\n([A-Za-z]+)",
        expand=False
    )
)

# conversion formal a objeto datetime
df_soccerway_completo["fecha_partido"] = pd.to_datetime(
    df_soccerway_completo["fecha_limpia"],
    format="%d.%m.%Y",
    errors="coerce"
)

# verificacion de la limpieza de fechas y distribucion de estados
print(
    "Fechas inválidas:",
    df_soccerway_completo["fecha_partido"]
    .isna()
    .sum()
)

print("\nEstados especiales:")
display(
    df_soccerway_completo[
        "estado_especial"
    ].value_counts(dropna=False)
)

Fechas inválidas: 0

Estados especiales:


estado_especial
<NA>    187
AET       1
Awrd      1
Pen       1
Name: count, dtype: Int64

### Celda 15.28. Reconstrucción de marcadores principales y gestión de definición por penales
Para gestionar adecuadamente los partidos definidos en tanda de penaltis (donde la cadena extraída de Soccerway incluye saltos de línea y el marcador parcial entre paréntesis, ej. `0\n(4)`), reconstruyo la información utilizando los registros brutos (`registros_2017_2018`). Preservo la cadena original en `goles_local_fuente` y `goles_visitante_fuente` y utilizo expresiones regulares (`^\s*(\d+)`) para extraer únicamente los goles del tiempo reglamentario/prórroga antes de castear a `Int64`.

In [75]:
import pandas as pd

# reconstruccion de DataFrame a partir de los registros crudos capturados por Selenium
df_soccerway_raw_final = pd.DataFrame(
    registros_2017_2018
)

# conservar marcador textual original como respaldo auditable
df_soccerway_completo["goles_local_fuente"] = (
    df_soccerway_raw_final["goles_local"]
)

df_soccerway_completo["goles_visitante_fuente"] = (
    df_soccerway_raw_final["goles_visitante"]
)

# extraer el marcador principal (tiempo reglamentario/prórroga)
df_soccerway_completo["goles_local"] = pd.to_numeric(
    df_soccerway_completo[
        "goles_local_fuente"
    ]
    .astype(str)
    .str.extract(r"^\s*(\d+)", expand=False),
    errors="coerce"
).astype("Int64")

df_soccerway_completo["goles_visitante"] = pd.to_numeric(
    df_soccerway_completo[
        "goles_visitante_fuente"
    ]
    .astype(str)
    .str.extract(r"^\s*(\d+)", expand=False),
    errors="coerce"
).astype("Int64")

# verificacion de parsing de marcadores tras la extraccion
print(
    "Marcadores locales nulos:",
    df_soccerway_completo["goles_local"].isna().sum()
)

print(
    "Marcadores visitantes nulos:",
    df_soccerway_completo["goles_visitante"].isna().sum()
)

display(
    df_soccerway_completo[
        [
            "fecha_partido",
            "equipo_local",
            "equipo_visitante",
            "goles_local_fuente",
            "goles_local",
            "goles_visitante_fuente",
            "goles_visitante"
        ]
    ].head(10)
)

Marcadores locales nulos: 0
Marcadores visitantes nulos: 0


,fecha_partido,equipo_local,equipo_visitante,goles_local_fuente,goles_local,goles_visitante_fuente,goles_visitante
0,2018-05-19,CA Independiente de La Chorrera,Tauro,1,1,0,0
1,2018-05-12,CA Independiente de La Chorrera,San Francisco FC,3,3,1,1
2,2018-05-11,Árabe Unido,Tauro,2,2,2,2
3,2018-05-05,Tauro,Árabe Unido,2,2,0,0
4,2018-05-04,San Francisco FC,CA Independiente de La Chorrera,1,1,0,0
5,2018-04-28,Alianza,Chorrillo,0,0,1,1
6,2018-04-28,Árabe Unido,San Francisco FC,1,1,1,1
7,2018-04-28,Plaza Amador,CA Independiente de La Chorrera,0,0,0,0
8,2018-04-28,Tauro,Sporting San Miguelito,3,3,0,0
9,2018-04-27,Santa Gema,Atlético Veragüense,1,1,0,0


In [78]:
df_soccerway_completo["penales_local"] = pd.to_numeric(
    df_soccerway_completo[
        "goles_local_fuente"
    ]
    .astype(str)
    .str.extract(r"\((\d+)\)", expand=False),
    errors="coerce"
).astype("Int64")

df_soccerway_completo["penales_visitante"] = pd.to_numeric(
    df_soccerway_completo[
        "goles_visitante_fuente"
    ]
    .astype(str)
    .str.extract(r"\((\d+)\)", expand=False),
    errors="coerce"
).astype("Int64")

In [79]:
display(
    df_soccerway_completo.loc[
        df_soccerway_completo[
            "estado_especial"
        ].notna(),
        [
            "source_match_id",
            "fecha_partido",
            "estado_especial",
            "equipo_local",
            "equipo_visitante",
            "goles_local",
            "goles_visitante",
            "penales_local",
            "penales_visitante"
        ]
    ]
)

,source_match_id,fecha_partido,estado_especial,equipo_local,equipo_visitante,goles_local,goles_visitante,penales_local,penales_visitante
2,tvO2cwtb,2018-05-11,AET,Árabe Unido,Tauro,2,2,<NA>,<NA>
70,YeopAmAn,2018-02-03,Awrd,Alianza,Tauro,3,0,<NA>,<NA>
97,j36Txm26,2017-11-25,Pen,Plaza Amador,Árabe Unido,0,1,4,5


In [80]:
df_soccerway_completo["elegible_modelado"] = True

df_soccerway_completo.loc[
    df_soccerway_completo[
        "estado_especial"
    ].eq("Awrd"),
    "elegible_modelado"
] = False

In [81]:
print(
    "Partidos históricos:",
    len(df_soccerway_completo)
)

print(
    "Partidos elegibles para modelado:",
    df_soccerway_completo["elegible_modelado"].sum()
)

print(
    "Partidos excluidos del modelado:",
    (~df_soccerway_completo["elegible_modelado"]).sum()
)

Partidos históricos: 190
Partidos elegibles para modelado: 189
Partidos excluidos del modelado: 1


In [82]:
display(
    df_soccerway_completo.loc[
        ~df_soccerway_completo["elegible_modelado"],
        [
            "source_match_id",
            "fecha_partido",
            "estado_especial",
            "equipo_local",
            "equipo_visitante",
            "goles_local",
            "goles_visitante"
        ]
    ]
)

,source_match_id,fecha_partido,estado_especial,equipo_local,equipo_visitante,goles_local,goles_visitante
70,YeopAmAn,2018-02-03,Awrd,Alianza,Tauro,3,0


### Celda 15.30. Identificación de los encuentros cargados adicionalmente
Comparo los identificadores de los 109 encuentros disponibles en el feed inicial con los 190 registros obtenidos después de cargar todos los resultados históricos. Esta comparación permite identificar exclusivamente los partidos añadidos mediante la opción de carga adicional y verificar que correspondan a los encuentros faltantes de la fase regular del Apertura 2017.

In [90]:
# IDs presentes en el feed inicial
ids_iniciales = set(
    df_soccerway_raw["source_match_id"]
    .dropna()
    .astype(str)
)

# IDs presentes después de cargar todos los partidos
ids_completos = set(
    df_soccerway_completo["source_match_id"]
    .dropna()
    .astype(str)
)

# encuentros incorporados mediante "Show more matches"
ids_adicionales = (
    ids_completos
    - ids_iniciales
)

print(
    "Partidos en feed inicial:",
    len(ids_iniciales)
)

print(
    "Partidos en extracción completa:",
    len(ids_completos)
)

print(
    "Partidos cargados adicionalmente:",
    len(ids_adicionales)
)

Partidos en feed inicial: 109
Partidos en extracción completa: 190
Partidos cargados adicionalmente: 81


### Celda 15.31 Auditoría cronológica del subconjunto de encuentros adicionales
Filtro el DataFrame consolidado `df_soccerway_completo` utilizando el conjunto de identificadores `ids_adicionales` para aislar los 81 partidos cargados dinámicamente tras la interacción con Selenium. Ordeno el subconjunto cronológicamente por `fecha_partido` para verificar la continuidad del rango temporal recuperado (confirmando la extracción completa del inicio del Torneo Apertura 2017) y despliego las primeras y últimas 10 filas para una revisión visual inmediata.

In [91]:
# Aislamiento y ordenamiento cronológico de los 81 partidos adicionados
df_adicionales = (
    df_soccerway_completo[
        df_soccerway_completo[
            "source_match_id"
        ].astype(str).isin(ids_adicionales)
    ]
    .copy()
    .sort_values("fecha_partido")
)

# Verificación de volumen y límites temporales
print(
    "Cantidad de partidos adicionales:",
    len(df_adicionales)
)

print(
    "Primera fecha adicional:",
    df_adicionales[
        "fecha_partido"
    ].min()
)

print(
    "Última fecha adicional:",
    df_adicionales[
        "fecha_partido"
    ].max()
)

# Muestra de los primeros 10 partidos recuperados (inicio de temporada)
display(
    df_adicionales[
        [
            "source_match_id",
            "fecha_partido",
            "equipo_local",
            "equipo_visitante",
            "goles_local",
            "goles_visitante"
        ]
    ].head(10)
)

# Muestra de los últimos 10 partidos recuperados del bloque adicional
display(
    df_adicionales[
        [
            "source_match_id",
            "fecha_partido",
            "equipo_local",
            "equipo_visitante",
            "goles_local",
            "goles_visitante"
        ]
    ].tail(10)
)

Cantidad de partidos adicionales: 81
Primera fecha adicional: 2017-07-14 00:00:00
Última fecha adicional: 2017-10-28 00:00:00


,source_match_id,fecha_partido,equipo_local,equipo_visitante,goles_local,goles_visitante
189,KCVWv62m,2017-07-14,San Francisco FC,Árabe Unido,0,2
188,CSqtyO1C,2017-07-14,Tauro,CA Independiente de La Chorrera,1,1
187,Yyoxx4n6,2017-07-15,Plaza Amador,Chorrillo,3,0
186,EZUzvQHg,2017-07-16,Atlético Veragüense,Santa Gema,2,1
185,hQTvwpXa,2017-07-16,Sporting San Miguelito,Alianza,1,3
184,htI2hPWn,2017-07-21,CA Independiente de La Chorrera,San Francisco FC,1,0
183,6kqpzrHI,2017-07-21,Árabe Unido,Santa Gema,2,1
182,KhmlZ1WO,2017-07-22,Atlético Veragüense,Sporting San Miguelito,1,0
181,McRWco9P,2017-07-22,Chorrillo,Tauro,2,0
180,O6Dbg5Ht,2017-07-22,Alianza,Plaza Amador,1,2


,source_match_id,fecha_partido,equipo_local,equipo_visitante,goles_local,goles_visitante
118,OOCVZYxs,2017-10-13,Chorrillo,Alianza,2,1
117,pv1vYC7g,2017-10-14,Plaza Amador,Árabe Unido,0,1
116,2FBZYhhm,2017-10-14,Tauro,Sporting San Miguelito,2,0
115,bs5rXWMa,2017-10-15,CA Independiente de La Chorrera,Atlético Veragüense,0,2
114,Wf3jVAiC,2017-10-20,Santa Gema,CA Independiente de La Chorrera,1,1
113,h44nWjx6,2017-10-20,Árabe Unido,Alianza,2,1
112,lpobynEo,2017-10-21,Atlético Veragüense,Chorrillo,1,1
111,zscGC2yo,2017-10-21,Plaza Amador,Tauro,1,0
110,bHZnUU6I,2017-10-21,Sporting San Miguelito,San Francisco FC,1,0
109,QRZRU1cT,2017-10-28,Atlético Veragüense,Árabe Unido,2,1


### Celda 15.32 Recuperación y limpieza de competiciones para los 109 partidos del feed inicial
Dado que el feed inicial (`df_soccerway_raw`) albergaba la clasificación precisa de competiciones extraída directamente de los bloques jerárquicos de la interfaz, rescato dicho mapeo para los 109 encuentros iniciales. Limpio la cadena removiendo prefijos geográficos (como `"PANAMA: "`) para homologar el texto y despliego la distribución de partidos por competición recuperada.

In [92]:
import pandas as pd

# Extracción de la asignación de competición desde el DataFrame del feed inicial
competicion_inicial = (
    df_soccerway_raw[
        [
            "source_match_id",
            "competicion_fuente"
        ]
    ]
    .copy()
)

# Limpieza y estandarización de la cadena de texto de la competición
competicion_inicial[
    "competicion_fuente"
] = (
    competicion_inicial[
        "competicion_fuente"
    ]
    .str.replace(
        "PANAMA: ",
        "",
        regex=False
    )
    .str.strip()
)

# Verificación de registros procesados y su desglose
print(
    "Registros iniciales clasificados:",
    len(competicion_inicial)
)

display(
    competicion_inicial[
        "competicion_fuente"
    ]
    .value_counts()
    .rename("partidos")
    .to_frame()
)

Registros iniciales clasificados: 109


,partidos
competicion_fuente,
LPF - Clausura,90
LPF - Apertura,9
LPF - Clausura - Play Offs,5
LPF - Apertura - Play Offs,5


### Celda 15.33 Reconstrucción final y asignación de competición
Elimino cualquier intento previo de clasificación e integro la información exacta de las fases: para los 109 partidos del feed inicial aplico una combinación izquierda (*left merge*) con validación `one_to_one` basada en `competicion_inicial`. Para los 81 partidos cargados adicionalmente (identificados en `ids_adicionales`), asigno explícitamente la etiqueta `"LPF - Apertura"` al corresponder a las jornadas faltantes de la fase regular de dicho torneo. Finalmente, confirmo que no existan registros sin competición.

In [99]:
# eliminar cualquier clasificacion incorrecta previa
df_soccerway_completo = (
    df_soccerway_completo
    .drop(
        columns=["competicion_fuente"],
        errors="ignore"
    )
)

# asociar la clasificacion conocida de los 109 partidos originales
df_soccerway_completo = (
    df_soccerway_completo
    .merge(
        competicion_inicial,
        on="source_match_id",
        how="left",
        validate="one_to_one"
    )
)

# los 81 partidos adicionales corresponden a la fase regular del Apertura 2017
mask_adicionales = (
    df_soccerway_completo[
        "source_match_id"
    ]
    .astype(str)
    .isin(ids_adicionales)
)

df_soccerway_completo.loc[
    mask_adicionales,
    "competicion_fuente"
] = "LPF - Apertura"

# verificacion de nulos y despliegue del conteo consolidado por fase
print(
    "Partidos sin competición:",
    df_soccerway_completo[
        "competicion_fuente"
    ].isna().sum()
)

display(
    df_soccerway_completo[
        "competicion_fuente"
    ]
    .value_counts()
    .rename("partidos")
    .to_frame()
)

Partidos sin competición: 0


,partidos
competicion_fuente,
LPF - Clausura,90
LPF - Apertura,90
LPF - Clausura - Play Offs,5
LPF - Apertura - Play Offs,5


### Celda 15.34 Comprobación matemática y validación de cobertura por fase
Estructuro la tabla de auditoría `tabla_cobertura` contrastando las frecuencias observadas en `competicion_fuente` con los valores esperados según el formato oficial de la LPF 2017/2018 (90 partidos en la fase regular y 5 en playoffs para el Apertura, e idéntica distribución para el Clausura). La columna booleana `consistente` y la validación `.all()` permiten certificar la completitud matemática de los 190 partidos..

In [98]:
# Diccionario con conteos observados por fase
cobertura_observada = {
    "Apertura - Fase regular": (
        df_soccerway_completo[
            "competicion_fuente"
        ] == "LPF - Apertura"
    ).sum(),

    "Apertura - Playoffs": (
        df_soccerway_completo[
            "competicion_fuente"
        ] == "LPF - Apertura - Play Offs"
    ).sum(),

    "Clausura - Fase regular": (
        df_soccerway_completo[
            "competicion_fuente"
        ] == "LPF - Clausura"
    ).sum(),

    "Clausura - Playoffs": (
        df_soccerway_completo[
            "competicion_fuente"
        ] == "LPF - Clausura - Play Offs"
    ).sum()
}

# Construcción del DataFrame comparativo
tabla_cobertura = pd.DataFrame({
    "componente": list(cobertura_observada.keys()),
    "observados": list(cobertura_observada.values()),
    "esperados": [90, 5, 90, 5]
})

# Validación de consistencia booleana
tabla_cobertura["consistente"] = (
    tabla_cobertura["observados"]
    == tabla_cobertura["esperados"]
)

display(tabla_cobertura)

# Impresión de resultados globales
print(
    "Cobertura total:",
    tabla_cobertura[
        "observados"
    ].sum()
)

print(
    "Todos los componentes consistentes:",
    tabla_cobertura[
        "consistente"
    ].all()
)

,componente,observados,esperados,consistente
0,Apertura - Fase regular,90,90,True
1,Apertura - Playoffs,5,5,True
2,Clausura - Fase regular,90,90,True
3,Clausura - Playoffs,5,5,True


Cobertura total: 190
Todos los componentes consistentes: True


In [95]:
df_soccerway_completo["torneo"] = np.where(
    df_soccerway_completo[
        "competicion_fuente"
    ].str.contains(
        "Apertura",
        case=False,
        na=False
    ),
    "Apertura",
    "Clausura"
)

df_soccerway_completo["tipo_fase"] = np.where(
    df_soccerway_completo[
        "competicion_fuente"
    ].str.contains(
        "Play Offs",
        case=False,
        na=False
    ),
    "Playoffs",
    "Fase regular"
)

tabla_cobertura_final = (
    df_soccerway_completo
    .groupby(
        ["torneo", "tipo_fase"]
    )
    .size()
    .rename("partidos")
    .reset_index()
)

display(tabla_cobertura_final)

,torneo,tipo_fase,partidos
0,Apertura,Fase regular,90
1,Apertura,Playoffs,5
2,Clausura,Fase regular,90
3,Clausura,Playoffs,5


### Celda 15.35 Derivación de la variable objetivo (resultado del encuentro)
Utilizo `np.select` para evaluar las condiciones sobre los marcadores principales (`goles_local` y `goles_visitante`) y construir la variable objetivo categórica `resultado` (`"L"` para victoria local, `"E"` para empate y `"V"` para victoria visitante). Esta derivación establece el *target* directo utilizado por los modelos de machine learning para la predicción de encuentros históricos, verificando que no existan valores inconsistentes (`"ERROR"`).

In [97]:
# Definicion de condiciones logicas para el resultado del tiempo reglamentario
condiciones_1718 = [
    df_soccerway_completo["goles_local"]
    > df_soccerway_completo["goles_visitante"],

    df_soccerway_completo["goles_local"]
    == df_soccerway_completo["goles_visitante"],

    df_soccerway_completo["goles_local"]
    < df_soccerway_completo["goles_visitante"]
]

# Asignacion categorica de la variable objetivo
df_soccerway_completo["resultado"] = np.select(
    condiciones_1718,
    ["L", "E", "V"],
    default="ERROR"
)

# Auditoria de distribucion y validacion de errores
print("Distribución 2017/2018:")
display(
    df_soccerway_completo["resultado"]
    .value_counts()
    .rename("partidos")
    .to_frame()
)

print(
    "Errores de clasificación:",
    (df_soccerway_completo["resultado"] == "ERROR").sum()
)

print(
    "Total clasificado:",
    df_soccerway_completo["resultado"]
    .isin(["L", "E", "V"])
    .sum()
)

Distribución 2017/2018:


,partidos
resultado,
L,82
E,65
V,43


Errores de clasificación: 0
Total clasificado: 190


## Celda 16. Integración del conjunto histórico de la LPF
Una vez validada la temporada complementaria 2017/2018, construyo un esquema común entre los registros procedentes de FootyStats y Soccerway. La integración se limita a variables verificables en ambas fuentes, preservando por separado las estadísticas avanzadas disponibles exclusivamente en FootyStats.Este procedimiento evita imputar información inexistente en la fuente complementaria y permite mantener la trazabilidad de cada registro mediante la variable de procedencia.

### Celda 16.1 Preparación del conjunto base de FootyStats para el esquema común
Ajusto el DataFrame `df_complete` para construir `df_footystats_base` bajo la estructura homologada. Se convierte la estampa de tiempo UTC de FootyStats a la zona horaria local (`America/Panama`), eliminando la información de zona horaria (`tz_localize(None)`) y normalizando la fecha a medianoche (`.normalize()`) para garantizar una coincidencia cronológica exacta con los registros de Soccerway. Adicionalmente, se inicializan con `pd.NA` aquellas columnas exclusivas de la fuente complementaria y se marca el indicador `elegible_modelado = True`.

In [100]:
df_footystats_base = pd.DataFrame()

# Fecha local de Panamá para homologar con el calendario local de Soccerway
df_footystats_base["fecha_partido"] = (
    df_complete["fecha_partido"]
    .dt.tz_convert("America/Panama")
    .dt.tz_localize(None)
    .dt.normalize()
)

df_footystats_base["temporada"] = (
    df_complete["temporada"].astype("string")
)

df_footystats_base["home_team_name"] = (
    df_complete["home_team_name"]
    .astype("string")
    .str.strip()
)

df_footystats_base["away_team_name"] = (
    df_complete["away_team_name"]
    .astype("string")
    .str.strip()
)

df_footystats_base["home_team_goal_count"] = (
    df_complete["home_team_goal_count"]
    .astype("Int64")
)

df_footystats_base["away_team_goal_count"] = (
    df_complete["away_team_goal_count"]
    .astype("Int64")
)

df_footystats_base["total_goal_count"] = (
    df_complete["total_goal_count"]
    .astype("Int64")
)

df_footystats_base["resultado"] = (
    df_complete["resultado"].astype("string")
)

df_footystats_base["fuente"] = "FootyStats"

df_footystats_base["id_fuente"] = pd.NA
df_footystats_base["estado_especial"] = pd.NA
df_footystats_base["penales_local"] = pd.NA
df_footystats_base["penales_visitante"] = pd.NA
df_footystats_base["torneo"] = pd.NA
df_footystats_base["tipo_fase"] = pd.NA

# Los 2037 registros previamente validados son marcados como elegibles
df_footystats_base["elegible_modelado"] = True

df_footystats_base["archivo_origen"] = (
    df_complete["archivo_origen"].astype("string")
)

print(
    "Registros FootyStats preparados:",
    len(df_footystats_base)
)

display(
    df_footystats_base.head(10)
)

Registros FootyStats preparados: 2037


,fecha_partido,temporada,home_team_name,away_team_name,home_team_goal_count,away_team_goal_count,total_goal_count,resultado,fuente,id_fuente,estado_especial,penales_local,penales_visitante,torneo,tipo_fase,elegible_modelado,archivo_origen
0,2014-07-18,2014/2015,Árabe Unido,Alianza,0,1,1,V,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
1,2014-07-18,2014/2015,San Francisco FC,Tauro,0,0,0,E,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
2,2014-07-18,2014/2015,Atlético Chiriquí,Plaza Amador,0,0,0,E,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
3,2014-07-19,2014/2015,Chepo,Sporting San Miguelito,1,2,3,V,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
4,2014-07-19,2014/2015,Chorrillo,CA Independiente de La Chorrera,0,0,0,E,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
5,2014-07-25,2014/2015,CA Independiente de La Chorrera,Árabe Unido,1,0,1,L,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
6,2014-07-25,2014/2015,Tauro,Chorrillo,0,1,1,V,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
7,2014-07-26,2014/2015,Alianza,Chepo,1,0,1,L,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
8,2014-07-26,2014/2015,Plaza Amador,San Francisco FC,1,2,3,V,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
9,2014-07-27,2014/2015,Sporting San Miguelito,Atlético Chiriquí,2,0,2,L,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv


### Celda 16.2 Preparación del conjunto base de Soccerway bajo el esquema común
Estructuro el DataFrame `df_soccerway_base` a partir de `df_soccerway_completo`, adaptando y casteando cada campo al esquema estándar homologado previamente con FootyStats. Se fija la cadena literal `"2017/2018"` para la temporada, se asignan los nombres homologados de los equipos local y visitante, se recalcula la suma de goles totales (`total_goal_count`) y se preservan las variables específicas de procedencia (`id_fuente`, `estado_especial`, `penales_local`, `penales_visitante`, `torneo` y `tipo_fase`).

In [102]:
df_soccerway_base = pd.DataFrame()

# Normalizacion de la fecha a medianoche (coincidencia con calendario local)
df_soccerway_base["fecha_partido"] = (
    df_soccerway_completo["fecha_partido"]
    .dt.normalize()
)

# Asignacion de la temporada complementaria
df_soccerway_base["temporada"] = "2017/2018"

# Nombres homologados de los clubes participantes
df_soccerway_base["home_team_name"] = (
    df_soccerway_completo["equipo_local"]
    .astype("string")
    .str.strip()
)

df_soccerway_base["away_team_name"] = (
    df_soccerway_completo["equipo_visitante"]
    .astype("string")
    .str.strip()
)

# Marcadores principales en tiempo reglamentario / prórroga
df_soccerway_base["home_team_goal_count"] = (
    df_soccerway_completo["goles_local"]
    .astype("Int64")
)

df_soccerway_base["away_team_goal_count"] = (
    df_soccerway_completo["goles_visitante"]
    .astype("Int64")
)

# Calculo de goles totales por encuentro
df_soccerway_base["total_goal_count"] = (
    df_soccerway_base["home_team_goal_count"]
    + df_soccerway_base["away_team_goal_count"]
)

# Target / variable objetivo derivada (L, E, V)
df_soccerway_base["resultado"] = (
    df_soccerway_completo["resultado"]
    .astype("string")
)

# Metadatos de la fuente y trazabilidad
df_soccerway_base["fuente"] = "Soccerway"

df_soccerway_base["id_fuente"] = (
    df_soccerway_completo["source_match_id"]
    .astype("string")
)

df_soccerway_base["estado_especial"] = (
    df_soccerway_completo["estado_especial"]
    .astype("string")
)

# Variables exclusivas de tandas de penaltis
df_soccerway_base["penales_local"] = (
    df_soccerway_completo["penales_local"]
    .astype("Int64")
)

df_soccerway_base["penales_visitante"] = (
    df_soccerway_completo["penales_visitante"]
    .astype("Int64")
)

# Clasificación de torneo (Apertura/Clausura) y fase (Regular/Playoffs)
df_soccerway_base["torneo"] = (
    df_soccerway_completo["torneo"]
    .astype("string")
)

df_soccerway_base["tipo_fase"] = (
    df_soccerway_completo["tipo_fase"]
    .astype("string")
)

# Bandera de elegibilidad para modelado
df_soccerway_base["elegible_modelado"] = (
    df_soccerway_completo["elegible_modelado"]
    .astype(bool)
)

# Identificacion de la fuente HTML original
df_soccerway_base["archivo_origen"] = (
    "lpf_2017_2018_soccerway_completo.html"
)

# Verificacion de registros preparados
print(
    "Registros Soccerway preparados:",
    len(df_soccerway_base)
)

display(
    df_soccerway_base.head(10)
)

Registros Soccerway preparados: 190


,fecha_partido,temporada,home_team_name,away_team_name,home_team_goal_count,away_team_goal_count,total_goal_count,resultado,fuente,id_fuente,estado_especial,penales_local,penales_visitante,torneo,tipo_fase,elegible_modelado,archivo_origen
0,2018-05-19,2017/2018,CA Independiente de La Chorrera,Tauro,1,0,1,L,Soccerway,zLN36mXI,<NA>,<NA>,<NA>,Clausura,Playoffs,True,lpf_2017_2018_soccerway_completo.html
1,2018-05-12,2017/2018,CA Independiente de La Chorrera,San Francisco FC,3,1,4,L,Soccerway,YDObbJQi,<NA>,<NA>,<NA>,Clausura,Playoffs,True,lpf_2017_2018_soccerway_completo.html
2,2018-05-11,2017/2018,Árabe Unido,Tauro,2,2,4,E,Soccerway,tvO2cwtb,AET,<NA>,<NA>,Clausura,Playoffs,True,lpf_2017_2018_soccerway_completo.html
3,2018-05-05,2017/2018,Tauro,Árabe Unido,2,0,2,L,Soccerway,fNPfaaBo,<NA>,<NA>,<NA>,Clausura,Playoffs,True,lpf_2017_2018_soccerway_completo.html
4,2018-05-04,2017/2018,San Francisco FC,CA Independiente de La Chorrera,1,0,1,L,Soccerway,hIoCzGYp,<NA>,<NA>,<NA>,Clausura,Playoffs,True,lpf_2017_2018_soccerway_completo.html
5,2018-04-28,2017/2018,Alianza,Chorrillo,0,1,1,V,Soccerway,vymYlbuU,<NA>,<NA>,<NA>,Clausura,Fase regular,True,lpf_2017_2018_soccerway_completo.html
6,2018-04-28,2017/2018,Árabe Unido,San Francisco FC,1,1,2,E,Soccerway,pEmUkvQN,<NA>,<NA>,<NA>,Clausura,Fase regular,True,lpf_2017_2018_soccerway_completo.html
7,2018-04-28,2017/2018,Plaza Amador,CA Independiente de La Chorrera,0,0,0,E,Soccerway,AeeHhtt5,<NA>,<NA>,<NA>,Clausura,Fase regular,True,lpf_2017_2018_soccerway_completo.html
8,2018-04-28,2017/2018,Tauro,Sporting San Miguelito,3,0,3,L,Soccerway,lnfLi0eB,<NA>,<NA>,<NA>,Clausura,Fase regular,True,lpf_2017_2018_soccerway_completo.html
9,2018-04-27,2017/2018,Santa Gema,Atlético Veragüense,1,0,1,L,Soccerway,6NlQjKAH,<NA>,<NA>,<NA>,Clausura,Fase regular,True,lpf_2017_2018_soccerway_completo.html


### Celda 16.3  Verificación de alineación de esquemas antes de la concatenación
Antes de unificar las fuentes en el DataFrame histórico consolidado, verifico estructuralmente que `df_footystats_base` y `df_soccerway_base` posean exactamente las mismas columnas y en el mismo orden. Esta comprobación garantiza que la concatenación por filas mediante `pd.concat` se realice de forma limpia y sin generar columnas redundantes ni desalineaciones de tipos.

In [103]:
# Comprobacion de coincidencia exacta de columnas y dimensiones
print(
    "Mismas columnas:",
    list(df_footystats_base.columns)
    == list(df_soccerway_base.columns)
)

print(
    "Número de columnas FootyStats:",
    df_footystats_base.shape[1]
)

print(
    "Número de columnas Soccerway:",
    df_soccerway_base.shape[1]
)

# Diagnostico en caso de discrepancias en los nombres de variables
if list(df_footystats_base.columns) != list(df_soccerway_base.columns):
    print("\nSolo en FootyStats:")
    print(
        set(df_footystats_base.columns)
        - set(df_soccerway_base.columns)
    )

    print("\nSolo en Soccerway:")
    print(
        set(df_soccerway_base.columns)
        - set(df_footystats_base.columns)
    )

Mismas columnas: True
Número de columnas FootyStats: 17
Número de columnas Soccerway: 17


### Celda 16.4 Integración definitiva del dataset histórico de la LPF y asignación de ID interno
Una vez confirmada la coincidencia exacta de los esquemas en la Celda 16.3, realizo la concatenación vertical (`pd.concat`) de `df_footystats_base` y `df_soccerway_base` para formar el conjunto consolidado `df_historico`. Ordeno el DataFrame cronológicamente por `fecha_partido`, `home_team_name` y `away_team_name`, e inserto en la primera posición la clave primaria estandarizada `registro_id` con el formato `LPF_XXXXX`. Finalmente, audito el total de registros por fuente y la elegibilidad para modelado.

In [104]:
# Concatenacion vertical de ambas fuentes tras confirmar la alineacion de columnas
df_historico = pd.concat(
    [
        df_footystats_base,
        df_soccerway_base
    ],
    ignore_index=True
)

# Ordenamiento cronologico e institucional
df_historico = (
    df_historico
    .sort_values(
        [
            "fecha_partido",
            "home_team_name",
            "away_team_name"
        ]
    )
    .reset_index(drop=True)
)

# Creacion e insercion del identificador unico interno
df_historico.insert(
    0,
    "registro_id",
    [
        f"LPF_{i:05d}"
        for i in range(1, len(df_historico) + 1)
    ]
)

# Auditoria de dimensiones e indicadores clave tras la unificacion
print(
    "Registros históricos integrados:",
    len(df_historico)
)

print(
    "Registros FootyStats:",
    (df_historico["fuente"] == "FootyStats").sum()
)

print(
    "Registros Soccerway:",
    (df_historico["fuente"] == "Soccerway").sum()
)

print(
    "Registros elegibles para modelado:",
    df_historico["elegible_modelado"].sum()
)

print(
    "Registros no elegibles:",
    (~df_historico["elegible_modelado"]).sum()
)

display(
    df_historico.head(10)
)

Registros históricos integrados: 2227
Registros FootyStats: 2037
Registros Soccerway: 190
Registros elegibles para modelado: 2226
Registros no elegibles: 1


,registro_id,fecha_partido,temporada,home_team_name,away_team_name,home_team_goal_count,away_team_goal_count,total_goal_count,resultado,fuente,id_fuente,estado_especial,penales_local,penales_visitante,torneo,tipo_fase,elegible_modelado,archivo_origen
0,LPF_00001,2014-07-18,2014/2015,Atlético Chiriquí,Plaza Amador,0,0,0,E,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
1,LPF_00002,2014-07-18,2014/2015,San Francisco FC,Tauro,0,0,0,E,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
2,LPF_00003,2014-07-18,2014/2015,Árabe Unido,Alianza,0,1,1,V,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
3,LPF_00004,2014-07-19,2014/2015,Chepo,Sporting San Miguelito,1,2,3,V,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
4,LPF_00005,2014-07-19,2014/2015,Chorrillo,CA Independiente de La Chorrera,0,0,0,E,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
5,LPF_00006,2014-07-25,2014/2015,CA Independiente de La Chorrera,Árabe Unido,1,0,1,L,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
6,LPF_00007,2014-07-25,2014/2015,Tauro,Chorrillo,0,1,1,V,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
7,LPF_00008,2014-07-26,2014/2015,Alianza,Chepo,1,0,1,L,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
8,LPF_00009,2014-07-26,2014/2015,Plaza Amador,San Francisco FC,1,2,3,V,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv
9,LPF_00010,2014-07-27,2014/2015,Sporting San Miguelito,Atlético Chiriquí,2,0,2,L,FootyStats,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,True,panama-lpf-matches-2014-to-2015-stats.csv


### Celda 16.5 Auditoría de calidad de datos e integridad matemática posintegración
Ejecuto una auditoría exhaustiva sobre `df_historico` para asegurar la ausencia de valores nulos en variables clave (`fecha_partido`, equipos y goles) y verificar la validez categórica del resultado (`"L"`, `"E"`, `"V"`). Adicionalmente, compruebo la consistencia aritmética del marcador confirmando que la suma de `home_team_goal_count` y `away_team_goal_count` sea idéntica a `total_goal_count` en todos los registros unificados.

In [106]:
# Verificacion de valores nulos criticos y validez del resultado
print("VALORES NULOS CRÍTICOS")

print(
    "Fechas nulas:",
    df_historico["fecha_partido"].isna().sum()
)

print(
    "Equipos locales nulos:",
    df_historico["home_team_name"].isna().sum()
)

print(
    "Equipos visitantes nulos:",
    df_historico["away_team_name"].isna().sum()
)

print(
    "Goles locales nulos:",
    df_historico["home_team_goal_count"].isna().sum()
)

print(
    "Goles visitantes nulos:",
    df_historico["away_team_goal_count"].isna().sum()
)

print(
    "Resultados inválidos:",
    (~df_historico["resultado"].isin(["L", "E", "V"])).sum()
)

# Auditoria de integridad matematica del marcador de goles
print("\nCONSISTENCIA DEL MARCADOR ")

consistencia_total = (
    df_historico["home_team_goal_count"]
    + df_historico["away_team_goal_count"]
    == df_historico["total_goal_count"]
)

print(
    "Marcadores consistentes:",
    consistencia_total.sum()
)

print(
    "Marcadores inconsistentes:",
    (~consistencia_total).sum()
)

VALORES NULOS CRÍTICOS
Fechas nulas: 0
Equipos locales nulos: 0
Equipos visitantes nulos: 0
Goles locales nulos: 0
Goles visitantes nulos: 0
Resultados inválidos: 0

CONSISTENCIA DEL MARCADOR 
Marcadores consistentes: 2227
Marcadores inconsistentes: 0


In [107]:
duplicados_integrados = df_historico.duplicated(
    subset=[
        "fecha_partido",
        "home_team_name",
        "away_team_name"
    ],
    keep=False
)

print(
    "Registros potencialmente duplicados:",
    duplicados_integrados.sum()
)

if duplicados_integrados.any():
    display(
        df_historico.loc[
            duplicados_integrados,
            [
                "registro_id",
                "fecha_partido",
                "temporada",
                "home_team_name",
                "away_team_name",
                "home_team_goal_count",
                "away_team_goal_count",
                "fuente"
            ]
        ].sort_values(
            [
                "fecha_partido",
                "home_team_name",
                "away_team_name"
            ]
        )
    )

Registros potencialmente duplicados: 0


### Celda 16.6. Distribución del conjunto histórico por temporada y fuente
Verifico la cantidad de encuentros disponibles para cada temporada después de integrar ambas fuentes. Esta comprobación permite confirmar la continuidad temporal del conjunto histórico y comprobar que la temporada 2017/2018 quedó incorporada correctamente entre los registros obtenidos mediante FootyStats.

In [108]:
# Conteo y agrupacion de partidos por temporada y fuente de origen
partidos_por_temporada = (
    df_historico
    .groupby(
        ["temporada", "fuente"],
        dropna=False
    )
    .size()
    .rename("partidos")
    .reset_index()
)

display(partidos_por_temporada)

# Verificación de la suma total de encontros consolidados
print(
    "\nTotal de partidos:",
    partidos_por_temporada["partidos"].sum()
)

,temporada,fuente,partidos
0,2014/2015,FootyStats,190
1,2015/2016,FootyStats,190
2,2016/2017,FootyStats,190
3,2017/2018,Soccerway,190
4,2018/2019,FootyStats,194
5,2020,FootyStats,92
6,2021,FootyStats,206
7,2022,FootyStats,206
8,2023,FootyStats,206
9,2024,FootyStats,206



Total de partidos: 2227


### Celda 16.7. Distribución de la variable objetivo integrada
Analizo la distribución global de la variable objetivo después de consolidar las fuentes históricas. Esta revisión permite conocer la proporción de victorias locales, empates y victorias visitantes que posteriormente será considerada durante la evaluación de los modelos de clasificación.

In [109]:
# Conteo absoluto de resultados globales (L, E, V)
distribucion_resultado = (
    df_historico["resultado"]
    .value_counts()
    .reindex(["L", "E", "V"])
)

# Calculo de la proporcion porcentual de cada resultado
porcentaje_resultado = (
    df_historico["resultado"]
    .value_counts(normalize=True)
    .reindex(["L", "E", "V"])
    * 100
)

# Estructuracion de la tabla resumida
tabla_resultado = pd.DataFrame({
    "partidos": distribucion_resultado,
    "porcentaje": porcentaje_resultado.round(2)
})

display(tabla_resultado)

# Verificacion del total de encuentros contabilizados
print(
    "Total:",
    tabla_resultado["partidos"].sum()
)

,partidos,porcentaje
resultado,,
L,862,38.71
E,699,31.39
V,666,29.91


Total: 2227


### Celda 16.8  Resumen técnico definitivo de la integración histórica
Consolido los indicadores cuantitativos de la integración en una tabla de auditoría final (`resumen_integracion`). Este resumen condensa el número total de encuentros unificados, la distribución por fuente (FootyStats vs. Soccerway), los registros elegibles para modelado y la verificación de consistencia (ausencia de valores nulos, inconsistencias en el marcador o duplicados potenciales), certificando que el dataset se encuentra listo para la fase de almacenamiento.

In [110]:
# Construcción del DataFrame sumario con los kpis finales de la integración
resumen_integracion = pd.DataFrame({
    "indicador": [
        "Registros históricos",
        "Registros FootyStats",
        "Registros Soccerway",
        "Registros elegibles para modelado",
        "Registros excluidos del modelado",
        "Fechas nulas",
        "Equipos locales nulos",
        "Equipos visitantes nulos",
        "Marcadores inconsistentes",
        "Duplicados potenciales"
    ],
    "valor": [
        len(df_historico),
        (df_historico["fuente"] == "FootyStats").sum(),
        (df_historico["fuente"] == "Soccerway").sum(),
        df_historico["elegible_modelado"].sum(),
        (~df_historico["elegible_modelado"]).sum(),
        df_historico["fecha_partido"].isna().sum(),
        df_historico["home_team_name"].isna().sum(),
        df_historico["away_team_name"].isna().sum(),
        (~consistencia_total).sum(),
        duplicados_integrados.sum()
    ]
})

display(resumen_integracion)

,indicador,valor
0,Registros históricos,2227
1,Registros FootyStats,2037
2,Registros Soccerway,190
3,Registros elegibles para modelado,2226
4,Registros excluidos del modelado,1
5,Fechas nulas,0
6,Equipos locales nulos,0
7,Equipos visitantes nulos,0
8,Marcadores inconsistentes,0
9,Duplicados potenciales,0


### Celda 16.9. Exportación de los conjuntos procesados
Exporto dos versiones del conjunto histórico a la carpeta de datos procesados (`PROCESSED_DIR`). La primera (`lpf_historico_integrado.csv`) conserva la totalidad de los partidos documentados y su información de trazabilidad. La segunda (`lpf_historico_modelado.csv`) filtra únicamente los registros identificados con `elegible_modelado = True`, garantizando un conjunto de datos limpio para las etapas posteriores de ingeniería de características y modelado predictivo.

In [111]:
# Asegurar la existencia del directorio de destino
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ARCHIVO_HISTORICO = (
    PROCESSED_DIR
    / "lpf_historico_integrado.csv"
)

ARCHIVO_MODELADO = (
    PROCESSED_DIR
    / "lpf_historico_modelado.csv"
)

# Exportación del dataset completo con trazabilidad total
df_historico.to_csv(
    ARCHIVO_HISTORICO,
    index=False,
    encoding="utf-8-sig"
)

# Filtrado y exportación del dataset exclusivo para modelado
df_modelado = (
    df_historico.loc[
        df_historico["elegible_modelado"]
    ]
    .copy()
    .reset_index(drop=True)
)

df_modelado.to_csv(
    ARCHIVO_MODELADO,
    index=False,
    encoding="utf-8-sig"
)

# Impresión de confirmación de rutas y conteos de registros
print("Dataset histórico:")
print(ARCHIVO_HISTORICO)
print("Registros:", len(df_historico))

print("\nDataset para modelado:")
print(ARCHIVO_MODELADO)
print("Registros:", len(df_modelado))

Dataset histórico:
C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\processed\lpf_historico_integrado.csv
Registros: 2227

Dataset para modelado:
C:\Users\ADMIN003\OneDrive - Universidad Tecnológica de Panamá\Escritorio\II SEMESTRE 2026\TESIS\Sistema_Predictivo_LPF\data\processed\lpf_historico_modelado.csv
Registros: 2226
